# SARD-ILD Code base

## Cohort Construction
The following nine cells construct a cohort of patients with at least one of six systemic autoimmune rheumatic diseases.

In [33]:
# Imports (all libraries used throughout the project)
import os, re, pandas_gbq
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from textwrap import dedent

pd.set_option('display.max_colwidth', None)   # no truncation of long string columns
pd.set_option('display.width', 0)             # auto-detect width, no line-wrapping cutoff
pd.set_option('display.max_columns', None)    # show all columns


ModuleNotFoundError: No module named 'pandas_gbq'

In [2]:
# Configuration: BigQuery dataset and common visit concept IDs

CDR = os.environ["WORKSPACE_CDR"]  # e.g., "aou-...-cdr2024q4"
INPATIENT_VISIT_CONCEPT_ID = 9201
OUTPATIENT_VISIT_CONCEPT_ID = 9202



In [3]:
# Helper: expand cohort-builder "root" ICD codes to full non-standard source concept_id list:
#   - find the selected rank-1 nodes in cb_criteria (by concept_code pattern)
#   - expand along cb_criteria.path to include the full selectable, non-standard set underneath

def _sql_or_join(clauses):
    return " OR ".join(f"({c})" for c in clauses)

def _sanitize_code_pattern(p):
    # Convert user-friendly patterns to SQL LIKE patterns
    # "M32.x" -> "M32.%"; "714.0" stays "714.0"
    p = p.strip()
    if p.endswith(".x"):
        return p[:-2] + ".%"
    if p.endswith("x"):
        return p[:-1] + "%"
    return p

def get_cohort_builder_concept_ids_from_icd_roots(
    icd_code_roots,
    vocabularies=("ICD9CM", "ICD10CM"),
    require_rank1=True
):
    """
    Parameters
    ----------
    icd_code_roots : list[str]
        Root ICD code strings, e.g., ["714.x", "M05.x"] or ["710.0", "M32.x"].
        Suffix '.x' is treated as a wildcard (converted to '%').
    vocabularies : tuple[str]
        Restrict seeds to these vocabularies (non-standard source codes).
    require_rank1 : bool
        If True, only expand nodes explicitly selected in the builder (full_text like '%_rank1]%').
        If no seeds are found, the function automatically falls back to allow non-rank1 seeds
        to avoid returning an empty list.

    Returns
    -------
    List[int] of concept_id values (non-standard, selectable) suitable for condition_source_concept_id IN (...)
    """
    assert icd_code_roots and isinstance(icd_code_roots, (list, tuple)), "icd_code_roots must be a non-empty list of strings."
    code_patterns = [ _sanitize_code_pattern(p) for p in icd_code_roots ]

    # Build the seed query: locate rank-1 nodes with matching concept_code patterns in specified vocabularies
    seeds_where = _sql_or_join([f"con.concept_code LIKE '{pat}'" for pat in code_patterns])
    vocab_where = _sql_or_join([f"con.vocabulary_id = '{v}'" for v in vocabularies])
    rank1_clause = "AND cr.full_text LIKE '%_rank1]%'" if require_rank1 else ""

    seed_sql = f"""
      SELECT DISTINCT CAST(cr.id AS STRING) AS id, cr.concept_id, con.concept_code, con.vocabulary_id
      FROM `{CDR}.cb_criteria` cr
      JOIN `{CDR}.concept` con
        ON con.concept_id = cr.concept_id
      WHERE ({seeds_where})
        AND ({vocab_where})
        AND cr.is_standard = 0
        AND cr.is_selectable = 1
        {rank1_clause}
    """

    seeds_df = pandas_gbq.read_gbq(seed_sql, dialect="standard")
    if seeds_df.empty and require_rank1:
        # Fall back to allow seeds even if they were not stored as rank-1 nodes
        seed_sql_relaxed = seed_sql.replace("AND cr.full_text LIKE '%_rank1]%'", "")
        seeds_df = pandas_gbq.read_gbq(seed_sql_relaxed, dialect="standard")

    if seeds_df.empty:
        raise ValueError(
            f"No seed cb_criteria rows found for patterns {code_patterns} in vocabularies {vocabularies}."
        )

    # Expand via cb_criteria.path to get the full non-standard selectable set (same as your original model)
    seeds_ids = "', '".join(seeds_df["id"].astype(str).tolist())
    expand_sql = f"""
      SELECT DISTINCT c.concept_id
      FROM `{CDR}.cb_criteria` c
      JOIN (SELECT CAST(cr.id AS STRING) AS id
            FROM `{CDR}.cb_criteria` cr
            WHERE CAST(cr.id AS STRING) IN ('{seeds_ids}')
      ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
         OR  c.path LIKE CONCAT('%.', a.id)
         OR  c.path LIKE CONCAT(a.id, '.%')
         OR  c.path = a.id)
      WHERE c.is_standard = 0
        AND c.is_selectable = 1
    """
    expanded = pandas_gbq.read_gbq(expand_sql, dialect="standard")
    concept_ids = sorted(set(int(x) for x in expanded["concept_id"].tolist()))
    if not concept_ids:
        raise ValueError("Expanded criteria produced an empty concept list; please verify input root codes.")
    return concept_ids



In [4]:
# Person-level filter builders

def _criteria_subselect_sql_from_concept_ids(concept_ids):
    # Small helper to embed the expanded concept_id list as a subselect
    ids_csv = ",".join(str(int(x)) for x in concept_ids)
    return f"""
      SELECT DISTINCT c.concept_id
      FROM `{CDR}.cb_criteria` c
      JOIN (
        SELECT CAST(cr.id AS STRING) AS id
        FROM `{CDR}.cb_criteria` cr
        WHERE cr.concept_id IN ({ids_csv})
          AND cr.full_text LIKE '%_rank1]%'
      ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
         OR  c.path LIKE CONCAT('%.', a.id)
         OR  c.path LIKE CONCAT(a.id, '.%')
         OR  c.path = a.id)
      WHERE c.is_standard = 0
        AND c.is_selectable = 1
    """

def _unnest_int_list(ints):
    return "UNNEST([" + ",".join(str(int(x)) for x in ints) + "])"

def person_filter_simple_gap_sql(expanded_concept_ids, min_days, max_days=None):
    """
    ≥2 qualifying events separated by [min_days, max_days] days.

    Identifies persons with at least two non-standard qualifying condition
    events (from cb_search_all_events) whose entry_date values are separated
    by at least min_days (and at most max_days, if specified).
    """

    crit_unnest = _unnest_int_list(expanded_concept_ids)
    later_bound = f"AND t2.entry_date <= DATE_ADD(t1.entry_date, INTERVAL {int(max_days)} DAY)" if max_days is not None else ""
    return f"""
      EXISTS (
        SELECT 1
        FROM `{CDR}.cb_search_all_events` t1
        WHERE t1.person_id = c_occurrence.person_id
          AND t1.concept_id IN {crit_unnest}
          AND t1.is_standard = 0
          AND EXISTS (
            SELECT 1
            FROM `{CDR}.cb_search_all_events` t2
            WHERE t2.person_id = t1.person_id
              AND t2.concept_id IN {crit_unnest}
              AND t2.is_standard = 0
              AND t2.entry_date >= DATE_ADD(t1.entry_date, INTERVAL {int(min_days)} DAY)
              {later_bound}
          )
      )
    """

def person_filter_simple_gap_with_index_meds_sql(expanded_concept_ids, min_days, med_concept_ids, med_window_days=None):
    """
    ≥2 qualifying events separated by at least min_days AND
    ≥1 qualifying medication on/within med_window_days after ANY qualifying code date.

    Qualifying events are non-standard disease events from cb_search_all_events.

    Med timing:
      - If med_window_days is None: medication date must be on/after some qualifying event date.
      - If med_window_days is int: medication date must fall within [event_date, event_date + med_window_days]
        for at least one qualifying event date.
    """

    crit_unnest = _unnest_int_list(expanded_concept_ids)

    # NOTE: anchor_expr below is an event date (not "first code date"):
    med_clause_any_event = med_exists_clause_sql_for_person(
        med_concept_ids,
        person_id_expr="e.person_id",
        anchor_expr="e.entry_date",
        window_days=med_window_days
    )

    return f"""
      EXISTS (
        WITH disease_events AS (
          SELECT person_id, entry_date
          FROM `{CDR}.cb_search_all_events`
          WHERE concept_id IN {crit_unnest}
            AND is_standard = 0
        ),
        qualifying_people AS (
          SELECT DISTINCT e1.person_id
          FROM disease_events e1
          WHERE EXISTS (
            SELECT 1
            FROM disease_events e2
            WHERE e2.person_id = e1.person_id
              AND e2.entry_date >= DATE_ADD(e1.entry_date, INTERVAL {int(min_days)} DAY)
          )
        )
        SELECT 1
        FROM disease_events e
        JOIN qualifying_people qp
          ON qp.person_id = e.person_id
        WHERE e.person_id = c_occurrence.person_id
        {med_clause_any_event}
      )
    """


In [5]:
# Inpatient and outpatient filter
def person_filter_inpatient_or_two_outpatient_sql(expanded_concept_ids, min_days, max_days=None):
    """
    Path A: ≥1 inpatient qualifying code; OR
    Path B: ≥2 outpatient qualifying codes separated by [min_days, max_days].
    """
    crit_unnest = _unnest_int_list(expanded_concept_ids)
    later_bound = f"AND e2.entry_date <= DATE_ADD(e1.entry_date, INTERVAL {int(max_days)} DAY)" if max_days is not None else ""
    return f"""
      EXISTS (
        SELECT 1
        FROM `{CDR}.cb_search_all_events` e1
        JOIN `{CDR}.visit_occurrence` v1
          ON v1.visit_occurrence_id = e1.visit_occurrence_id
        WHERE e1.person_id = c_occurrence.person_id
          AND e1.concept_id IN {crit_unnest}
          AND e1.is_standard = 0
          AND (
            -- A) At least one inpatient qualifying event
            v1.visit_concept_id = {int(INPATIENT_VISIT_CONCEPT_ID)}

            OR

            -- B) Two outpatient qualifying events separated by [min_days, max_days]
            EXISTS (
              SELECT 1
              FROM `{CDR}.cb_search_all_events` e2
              JOIN `{CDR}.visit_occurrence` v2
                ON v2.visit_occurrence_id = e2.visit_occurrence_id
              WHERE e2.person_id = e1.person_id
                AND e2.concept_id IN {crit_unnest}
                AND e2.is_standard = 0
                AND v1.visit_concept_id = {int(OUTPATIENT_VISIT_CONCEPT_ID)}
                AND v2.visit_concept_id = {int(OUTPATIENT_VISIT_CONCEPT_ID)}
                AND e2.entry_date >= DATE_ADD(e1.entry_date, INTERVAL {int(min_days)} DAY)
                {later_bound}
            )
          )
      )
    """

def person_filter_inpatient_or_two_outpatient_with_index_meds_sql(expanded_concept_ids, min_days, max_days, med_concept_ids, med_window_days=None):
    """
    Path A: ≥1 inpatient qualifying code; OR
    Path B: ≥2 outpatient qualifying codes separated by [min_days, max_days].
    AND ≥1 qualifying medication on/within med_window_days after ANY qualifying code date.

    Qualifying events are non-standard disease events from cb_search_all_events.
    """

    crit_unnest = _unnest_int_list(expanded_concept_ids)
    later_bound = f"AND e2.entry_date <= DATE_ADD(e1.entry_date, INTERVAL {int(max_days)} DAY)"

    # IMPORTANT: med is anchored to each event date (not first code date)
    med_clause_any_event = med_exists_clause_sql_for_person(
        med_concept_ids,
        person_id_expr="e.person_id",
        anchor_expr="e.entry_date",
        window_days=med_window_days
    )

    return f"""
      EXISTS (
        WITH disease_events AS (
          SELECT e.person_id, e.entry_date, v.visit_concept_id
          FROM `{CDR}.cb_search_all_events` e
          JOIN `{CDR}.visit_occurrence` v
            ON v.visit_occurrence_id = e.visit_occurrence_id
          WHERE e.concept_id IN {crit_unnest}
            AND e.is_standard = 0
        ),
        qualifying_people AS (
          SELECT DISTINCT person_id
          FROM (
            -- A) At least one inpatient qualifying event
            SELECT person_id
            FROM disease_events
            WHERE visit_concept_id = {int(INPATIENT_VISIT_CONCEPT_ID)}

            UNION DISTINCT

            -- B) Two outpatient qualifying events, separated by [min_days, max_days]
            SELECT DISTINCT e1.person_id
            FROM disease_events e1
            JOIN disease_events e2
              ON e1.person_id = e2.person_id
             AND e1.visit_concept_id = {int(OUTPATIENT_VISIT_CONCEPT_ID)}
             AND e2.visit_concept_id = {int(OUTPATIENT_VISIT_CONCEPT_ID)}
             AND e2.entry_date >= DATE_ADD(e1.entry_date, INTERVAL {int(min_days)} DAY)
             {later_bound}
          )
        )
        SELECT 1
        FROM disease_events e
        JOIN qualifying_people qp
          ON qp.person_id = e.person_id
        WHERE e.person_id = c_occurrence.person_id
        {med_clause_any_event}
      )
    """


In [6]:
# Medication EXISTS (optional) — standardized + descendant expansion
def med_exists_clause_sql(med_concept_ids, anchor_expr, window_days=None):
    """
    med_concept_ids : list[int]
        OMOP drug_concept_id values (may be standard or non-standard).
    anchor_expr : str
        SQL expression that resolves to a DATE (e.g., DATE(c_occurrence.condition_start_datetime)).
    window_days : Optional[int]
        None -> on/after anchor; int -> within [anchor, anchor + window_days].
    """
    if not med_concept_ids:
        return ""
    # Seeds literal (may include non-standard IDs)
    seeds = "UNNEST([" + ",".join(str(int(x)) for x in med_concept_ids) + "])"

    # Normalize to STANDARD concepts and include descendants so all clinical/branded forms count
    expanded_subselect = f"""
      SELECT DISTINCT ca.descendant_concept_id AS drug_concept_id
      FROM (
        -- standard seeds
        SELECT c.concept_id AS std_id
        FROM `{CDR}.concept` c
        JOIN {seeds} s ON c.concept_id = s
        WHERE c.standard_concept = 'S'

        UNION DISTINCT

        -- non-standard seeds -> 'Maps to' standard
        SELECT cr.concept_id_2 AS std_id
        FROM `{CDR}.concept_relationship` cr
        JOIN {seeds} s ON cr.concept_id_1 = s
        WHERE cr.relationship_id = 'Maps to'
      ) sm
      JOIN `{CDR}.concept_ancestor` ca
        ON ca.ancestor_concept_id = sm.std_id

      UNION DISTINCT
      -- include the ancestor itself
      SELECT sm.std_id AS drug_concept_id
      FROM (
        SELECT c.concept_id AS std_id
        FROM `{CDR}.concept` c
        JOIN {seeds} s ON c.concept_id = s
        WHERE c.standard_concept = 'S'
        UNION DISTINCT
        SELECT cr.concept_id_2 AS std_id
        FROM `{CDR}.concept_relationship` cr
        JOIN {seeds} s ON cr.concept_id_1 = s
        WHERE cr.relationship_id = 'Maps to'
      ) sm
    """

    if window_days is None:
        timing = f"DATE(de.drug_exposure_start_date) >= {anchor_expr}"
    else:
        timing = f"""
          DATE(de.drug_exposure_start_date)
          BETWEEN {anchor_expr}
              AND DATE_ADD({anchor_expr}, INTERVAL {int(window_days)} DAY)
        """

    return f"""
      AND EXISTS (
        SELECT 1
        FROM `{CDR}.drug_exposure` de
        WHERE de.person_id = c_occurrence.person_id
          AND EXISTS (
            SELECT 1
            FROM ({expanded_subselect}) m
            WHERE m.drug_concept_id = de.drug_concept_id
          )
          AND {timing}
      )
    """

def med_exists_clause_sql_for_person(med_concept_ids, person_id_expr, anchor_expr, window_days=None):
    """
    med_concept_ids : list[int]
        OMOP drug_concept_id values (may be standard or non-standard).
    person_id_expr : str
        SQL expression that resolves to the person_id in scope (e.g., 'person_id').
    anchor_expr : str
        SQL expression that resolves to a DATE (e.g., first_code_date).
    window_days : Optional[int]
        None -> on/after anchor; int -> within [anchor, anchor + window_days].
    """
    if not med_concept_ids:
        return ""
    seeds = "UNNEST([" + ",".join(str(int(x)) for x in med_concept_ids) + "])"

    expanded_subselect = f"""
      SELECT DISTINCT ca.descendant_concept_id AS drug_concept_id
      FROM (
        SELECT c.concept_id AS std_id
        FROM `{CDR}.concept` c
        JOIN {seeds} s ON c.concept_id = s
        WHERE c.standard_concept = 'S'

        UNION DISTINCT

        SELECT cr.concept_id_2 AS std_id
        FROM `{CDR}.concept_relationship` cr
        JOIN {seeds} s ON cr.concept_id_1 = s
        WHERE cr.relationship_id = 'Maps to'
      ) sm
      JOIN `{CDR}.concept_ancestor` ca
        ON ca.ancestor_concept_id = sm.std_id

      UNION DISTINCT
      SELECT sm.std_id AS drug_concept_id
      FROM (
        SELECT c.concept_id AS std_id
        FROM `{CDR}.concept` c
        JOIN {seeds} s ON c.concept_id = s
        WHERE c.standard_concept = 'S'
        UNION DISTINCT
        SELECT cr.concept_id_2 AS std_id
        FROM `{CDR}.concept_relationship` cr
        JOIN {seeds} s ON cr.concept_id_1 = s
        WHERE cr.relationship_id = 'Maps to'
      ) sm
    """

    if window_days is None:
        timing = f"DATE(de.drug_exposure_start_date) >= {anchor_expr}"
    else:
        timing = f"""
          DATE(de.drug_exposure_start_date)
          BETWEEN {anchor_expr}
              AND DATE_ADD({anchor_expr}, INTERVAL {int(window_days)} DAY)
        """

    return f"""
      AND EXISTS (
        SELECT 1
        FROM `{CDR}.drug_exposure` de
        WHERE de.person_id = {person_id_expr}
          AND EXISTS (
            SELECT 1
            FROM ({expanded_subselect}) m
            WHERE m.drug_concept_id = de.drug_concept_id
          )
          AND {timing}
      )
    """


In [7]:
# Main SELECT builder
def build_disease_select_sql(
    disease_name,
    expanded_concept_ids,
    person_filter_sql,
    med_concept_ids=None,
    med_window_days=None
):
    crit_unnest = _unnest_int_list(expanded_concept_ids)
    med_clause = med_exists_clause_sql(med_concept_ids or [], "DATE(c_occurrence.condition_start_datetime)", med_window_days)

    return f"""
    SELECT
      '{disease_name}' AS disease,
      c_occurrence.person_id,
      c_occurrence.condition_concept_id,
      c_standard_concept.concept_name AS standard_concept_name,
      c_standard_concept.concept_code AS standard_concept_code,
      c_standard_concept.vocabulary_id AS standard_vocabulary,
      c_occurrence.condition_start_datetime,
      c_occurrence.condition_end_datetime,
      c_occurrence.condition_type_concept_id,
      c_type.concept_name AS condition_type_concept_name,
      c_occurrence.stop_reason,
      c_occurrence.visit_occurrence_id,
      visit.concept_name AS visit_occurrence_concept_name,
      c_occurrence.condition_source_value,
      c_occurrence.condition_source_concept_id,
      c_source_concept.concept_name AS source_concept_name,
      c_source_concept.concept_code AS source_concept_code,
      c_source_concept.vocabulary_id AS source_vocabulary,
      c_occurrence.condition_status_source_value,
      c_occurrence.condition_status_concept_id,
      c_status.concept_name AS condition_status_concept_name
    FROM (
      SELECT *
      FROM `{CDR}.condition_occurrence` c_occurrence
      WHERE
        c_occurrence.condition_source_concept_id IN {crit_unnest}
        AND ({person_filter_sql})
        {med_clause}
    ) c_occurrence
    LEFT JOIN `{CDR}.concept` c_standard_concept
      ON c_occurrence.condition_concept_id = c_standard_concept.concept_id
    LEFT JOIN `{CDR}.concept` c_type
      ON c_occurrence.condition_type_concept_id = c_type.concept_id
    LEFT JOIN `{CDR}.visit_occurrence` v
      ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id
    LEFT JOIN `{CDR}.concept` visit
      ON v.visit_concept_id = visit.concept_id
    LEFT JOIN `{CDR}.concept` c_source_concept
      ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id
    LEFT JOIN `{CDR}.concept` c_status
      ON c_occurrence.condition_status_concept_id = c_status.concept_id
    """

def union_all_sql(blocks):
    return "\nUNION ALL\n".join(b.strip() for b in blocks if b and b.strip())



In [8]:
# Medication concept_id lists (OMOP drug_concept_id)
# RA
RA_MED_IDS = [
    1186087, 1119119, 1114375, 19014878, 19010482, 19028050, 1151789, 42899196,
    1777087, 937368, 1101898, 1305058, 1708880, 1314273, 964339
]

# SLE (contemporary approvals)
SLE_MED_IDS = [
    1092437,  # belimumab
    701470,   # anifrolumab
    1777087,  # hydroxychloroquine
    # (ocrelizumab intentionally omitted; marked "SLE?" in table)
]

# IM
IM_MED_IDS = [
    19014878, 19010482, 1777087, 1101898, 1305058, 1314273,
    19003999, 19068900,  # mycophenolate
    1310317,             # cyclophosphamide
    19117912             # IVIG
]

# Define root ICD code patterns per disease
DISEASE_ROOTS = {
    "RA":      ["714.x", "M05.x", "M06.x"],
    "SLE":     ["710.0", "M32.x"],
    "SSc":     ["710.1", "517.2", "M34.x"],
    "Sjögren": ["710.2", "M35.0"],
    "IM":      ["710.3", "710.4", "728.81", "M33.x", "M60.10"],
    "MCTD":    ["710.8", "M35.1"]
}

# Build select_blocks
# Expand ICD roots to non-standard source concept_ids (rank-1 seeds expanded along cb_criteria.path)
_expanded = {
    d: get_cohort_builder_concept_ids_from_icd_roots(roots)
    for d, roots in DISEASE_ROOTS.items()
}

# Person filter SQL per disease
# RA: 2+ codes, 7+ days apart; + meds within 365 days after diagnosis
_pf_ra  = person_filter_simple_gap_with_index_meds_sql(_expanded["RA"], min_days=7, med_concept_ids=RA_MED_IDS, med_window_days=365)

# SLE: 2+ codes, 30+ days apart; + ≥1 prescription (no upper window; on/after index)
_pf_sle = person_filter_simple_gap_with_index_meds_sql(_expanded["SLE"], min_days=30, med_concept_ids=SLE_MED_IDS, med_window_days=None)

# SSc: 1+ inpatient OR 2+ outpatient, 30+ days apart
_pf_ssc = person_filter_inpatient_or_two_outpatient_sql(_expanded["SSc"], min_days=30)

# Sjögren: 2+ codes, 4+ weeks apart (~28 days)
_pf_sj  = person_filter_simple_gap_sql(_expanded["Sjögren"], min_days=28)

# IM: 1+ inpatient OR 2+ outpatient, 30–365 days apart; + meds within 365 days
_pf_im  = person_filter_inpatient_or_two_outpatient_with_index_meds_sql(_expanded["IM"], min_days=30, max_days=365, med_concept_ids=IM_MED_IDS, med_window_days=365)

# MCTD: 1+ inpatient OR 2+ outpatient, 30–365 days apart
_pf_mctd = person_filter_inpatient_or_two_outpatient_sql(_expanded["MCTD"], min_days=30, max_days=365)

# Build the per-disease SELECT blocks (with med windows where specified)
select_blocks = [
    build_disease_select_sql(
        "RA", _expanded["RA"], _pf_ra,
        med_concept_ids=None, med_window_days=None  # within 365 days
    ),
    build_disease_select_sql(
        "SLE", _expanded["SLE"], _pf_sle,
        med_concept_ids=None, med_window_days=None  # on/after index; no upper bound per table
    ),
    build_disease_select_sql(
        "SSc", _expanded["SSc"], _pf_ssc
    ),
    build_disease_select_sql(
        "Sjögren", _expanded["Sjögren"], _pf_sj
    ),
    build_disease_select_sql(
        "IM", _expanded["IM"], _pf_im,
        med_concept_ids=None, med_window_days=None  # within 365 days
    ),
    build_disease_select_sql(
        "MCTD", _expanded["MCTD"], _pf_mctd
    ),
]


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


In [9]:
# UNION ALL, run, and preview

final_sql = union_all_sql(select_blocks)

df = pandas_gbq.read_gbq(
    final_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

print(f"Final row count: {len(df):,}")
df.head(10)


Downloading:   0%|          |

Final row count: 697,291


,disease,person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
0,MCTD,5569012,255334,Collagen disease,81573002,SNOMED,2016-09-22 00:00:00+00:00,NaT,0,No matching concept,None,30000000003610827,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
1,MCTD,2366362,255334,Collagen disease,81573002,SNOMED,2000-10-28 06:00:00+00:00,NaT,32821,EHR billing record,None,66000000009449070,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
2,MCTD,1143024,255334,Collagen disease,81573002,SNOMED,2013-10-13 00:00:00+00:00,NaT,32821,EHR billing record,None,15000000029650093,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
3,MCTD,3284911,255334,Collagen disease,81573002,SNOMED,2011-07-01 00:00:00+00:00,2011-07-01 11:59:59+00:00,44786629,Secondary Condition,None,37000000007264149,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
4,MCTD,5776109,255334,Collagen disease,81573002,SNOMED,2011-04-06 00:00:00+00:00,2011-04-06 11:59:59+00:00,44786627,Primary Condition,None,37000000001997178,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
5,MCTD,1680268,255334,Collagen disease,81573002,SNOMED,2012-09-29 00:00:00+00:00,NaT,0,No matching concept,None,30000000001224237,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
6,MCTD,6527462,255334,Collagen disease,81573002,SNOMED,2012-09-25 06:00:00+00:00,NaT,32821,EHR billing record,None,66000000069840979,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
7,MCTD,3815201,255334,Collagen disease,81573002,SNOMED,2015-10-14 00:00:00+00:00,NaT,0,No matching concept,None,30000000020966518,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
8,MCTD,2177047,255334,Collagen disease,81573002,SNOMED,2013-03-05 05:00:00+00:00,NaT,44786629,Secondary Condition,None,33000000000887644,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None
9,MCTD,1995495,255334,Collagen disease,81573002,SNOMED,2006-01-29 00:00:00+00:00,2006-01-29 11:59:59+00:00,44786627,Primary Condition,None,37000000009296831,Outpatient Visit,710.8,44825662,Other specified diffuse diseases of connective tissue,710.8,ICD9CM,None,<NA>,None


In [2]:
# Use a slimmer cohort_sql to reduce BigQuery planning complexity
final_sql_materialized = dedent(f"""
SELECT
  person_id,
  disease,
  condition_start_datetime
FROM ({final_sql}) AS cohort_rows
""")

NameError: name 'dedent' is not defined

## Descriptive Statistics and Comparative Analysis
The next 11 cells are dedicated to building summary tables of the overall SARD cohort and the interstitial lung disease (ILD) sub-cohort and to performing some initial comparative analyses.

In [11]:
# Defines ILD code roots and expands to cohort-builder non-standard concept_ids.
# Defines literal UNNEST(...) string for use in SQL.

ILD_ROOTS = ["515.x", "516.3", "516.8", "516.9", "J84.1", "J84.89", "J84.9"]
ILD_CONCEPT_IDS = get_cohort_builder_concept_ids_from_icd_roots(ILD_ROOTS)
ILD_UNNEST = "UNNEST([" + ",".join(str(int(x)) for x in ILD_CONCEPT_IDS) + "])"

# Builds a BigQuery UNNEST literal from a Python list of integers.
def _unnest_literal(int_list):
    return "UNNEST([" + ",".join(str(int(x)) for x in int_list) + "])"

# Combined medication seed list from existing lists (standard and non-standard IDs are acceptable).
ALL_MED_IDS_LIST = sorted(set(RA_MED_IDS + SLE_MED_IDS + IM_MED_IDS))
ALL_MED_IDS_UNNEST = _unnest_literal(ALL_MED_IDS_LIST)


Downloading: 100%|██████████|
Downloading: 100%|██████████|


In [4]:
# Summary table builder

def build_sard_summary_table(
    cohort_sql: str,
    definition_row: dict,
    include_ild_rows: bool = True,
    row_order: list[str] | None = None,
    debug: bool = False,
) -> pd.DataFrame:
    """
    Plan-safe summary table builder for All of Us SARD cohorts.

    Key changes vs original:
      - Single shared cohort/index CTE reused in every block
      - Conditional aggregation instead of many scalar subqueries
      - ILD "loose" definition rewritten to avoid cb_search_all_events self-join
      - Optional debug mode to print which block fails
      - Age computed as continuous years (days/365.25) to reduce "age heaping" bias
    """

    # -----------------------
    # Shared base CTE
    # -----------------------
    BASE = dedent(f"""
    WITH
    cohort AS (
      {cohort_sql}
    ),
    index_by_person AS (
      SELECT disease, person_id, MIN(DATE(condition_start_datetime)) AS index_date
      FROM cohort
      GROUP BY disease, person_id
    ),
    index_all_by_person AS (
      SELECT person_id, MIN(DATE(condition_start_datetime)) AS index_date
      FROM cohort
      GROUP BY person_id
    )
    """)

    def _read(sql: str) -> pd.DataFrame:
        return pandas_gbq.read_gbq(sql, dialect="standard")

    # -----------------------
    # AGE (continuous years)
    sql_row_age = dedent(f"""
    {BASE},
    p AS (
      SELECT person_id, birth_datetime
      FROM `{CDR}.person`
    ),
    dem AS (
      SELECT
        i.disease,
        i.person_id,
        SAFE_DIVIDE(DATE_DIFF(i.index_date, DATE(p.birth_datetime), DAY), 365.25) AS age_years
      FROM index_by_person i
      JOIN p USING (person_id)
    ),
    dem_all AS (
      SELECT
        a.person_id,
        SAFE_DIVIDE(DATE_DIFF(a.index_date, DATE(p.birth_datetime), DAY), 365.25) AS age_years
      FROM index_all_by_person a
      JOIN p USING (person_id)
    ),
    all_stats AS (
      SELECT AVG(age_years) AS mean_age, STDDEV_SAMP(age_years) AS sd_age
      FROM dem_all
    )
    SELECT
      'Age (years, mean, SD)' AS row_label,
      FORMAT('%.1f (%.1f)', (SELECT mean_age FROM all_stats), (SELECT sd_age FROM all_stats)) AS `All SARDs`,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='RA',      age_years, NULL)), STDDEV_SAMP(IF(disease='RA',      age_years, NULL))) AS RA,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='SLE',     age_years, NULL)), STDDEV_SAMP(IF(disease='SLE',     age_years, NULL))) AS SLE,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='SSc',     age_years, NULL)), STDDEV_SAMP(IF(disease='SSc',     age_years, NULL))) AS SSc,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='Sjögren', age_years, NULL)), STDDEV_SAMP(IF(disease='Sjögren', age_years, NULL))) AS `Sjögren`,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='IM',      age_years, NULL)), STDDEV_SAMP(IF(disease='IM',      age_years, NULL))) AS IM,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='MCTD',    age_years, NULL)), STDDEV_SAMP(IF(disease='MCTD',    age_years, NULL))) AS MCTD
    FROM dem
    """)

    # -----------------------
    # SEX (Female / Male)
    # -----------------------
    def _sex_row_sql(label: str, target: str) -> str:
        return dedent(f"""
        {BASE},
        p AS (
          SELECT person_id, COALESCE(sex_at_birth_concept_id, 0) AS sex_id
          FROM `{CDR}.person`
        ),
        concept_map AS (
          SELECT concept_id, concept_name FROM `{CDR}.concept`
        ),
        dem AS (
          SELECT i.disease, i.person_id, COALESCE(cm.concept_name, 'Unknown') AS sex
          FROM index_by_person i
          JOIN p USING (person_id)
          LEFT JOIN concept_map cm ON cm.concept_id = p.sex_id
        ),
        dem_all AS (
          SELECT a.person_id, COALESCE(cm.concept_name, 'Unknown') AS sex
          FROM index_all_by_person a
          JOIN p USING (person_id)
          LEFT JOIN concept_map cm ON cm.concept_id = p.sex_id
        ),
        denom AS (
          SELECT disease, COUNT(DISTINCT person_id) AS n_people
          FROM index_by_person
          GROUP BY disease
          UNION ALL
          SELECT 'All SARDs' AS disease, COUNT(DISTINCT person_id) AS n_people
          FROM index_all_by_person
        ),
        num_all AS (
          SELECT COUNT(DISTINCT person_id) AS n
          FROM dem_all
          WHERE sex = '{target}'
        )
        SELECT
          '{label}' AS row_label,
          FORMAT('%d (%.1f%%)',
                 (SELECT n FROM num_all),
                 100.0*SAFE_DIVIDE((SELECT n FROM num_all), (SELECT n_people FROM denom WHERE disease='All SARDs'))) AS `All SARDs`,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(disease='RA'      AND sex='{target}', person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(disease='RA'      AND sex='{target}', person_id, NULL)),
                                   (SELECT n_people FROM denom WHERE disease='RA'))) AS RA,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(disease='SLE'     AND sex='{target}', person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(disease='SLE'     AND sex='{target}', person_id, NULL)),
                                   (SELECT n_people FROM denom WHERE disease='SLE'))) AS SLE,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(disease='SSc'     AND sex='{target}', person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(disease='SSc'     AND sex='{target}', person_id, NULL)),
                                   (SELECT n_people FROM denom WHERE disease='SSc'))) AS SSc,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(disease='Sjögren' AND sex='{target}', person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(disease='Sjögren' AND sex='{target}', person_id, NULL)),
                                   (SELECT n_people FROM denom WHERE disease='Sjögren'))) AS `Sjögren`,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(disease='IM'      AND sex='{target}', person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(disease='IM'      AND sex='{target}', person_id, NULL)),
                                   (SELECT n_people FROM denom WHERE disease='IM'))) AS IM,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(disease='MCTD'    AND sex='{target}', person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(disease='MCTD'    AND sex='{target}', person_id, NULL)),
                                   (SELECT n_people FROM denom WHERE disease='MCTD'))) AS MCTD
        FROM dem
        """)

    sql_row_female = _sex_row_sql("Female sex (n, %)", "Female")
    sql_row_male   = _sex_row_sql("Male sex (n, %)",   "Male")

    # -----------------------
    # RACE (dynamic; fewer subqueries)
    # -----------------------
    sql_row_race_all = dedent(f"""
    {BASE},
    race_raw AS (
      SELECT person_id, answer AS race
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1586140
    ),
    race_disease AS (
      SELECT i.disease, i.person_id, COALESCE(rr.race, 'Prefer not to answer / missing') AS race
      FROM index_by_person i
      LEFT JOIN race_raw rr USING (person_id)
    ),
    race_all AS (
      SELECT a.person_id, COALESCE(rr.race, 'Prefer not to answer / missing') AS race
      FROM index_all_by_person a
      LEFT JOIN race_raw rr USING (person_id)
    ),
    levels AS (
      SELECT DISTINCT race FROM race_all
      UNION DISTINCT
      SELECT DISTINCT race FROM race_disease
    ),
    denom AS (
      SELECT disease, COUNT(DISTINCT person_id) AS n_people
      FROM index_by_person
      GROUP BY disease
      UNION ALL
      SELECT 'All SARDs' AS disease, COUNT(DISTINCT person_id) AS n_people
      FROM index_all_by_person
    ),
    counts_all AS (
      SELECT race, COUNT(DISTINCT person_id) AS n
      FROM race_all
      GROUP BY race
    )
    SELECT
      CONCAT('RACE: ', l.race) AS row_label,
      FORMAT('%d (%.1f%%)',
             COALESCE(ca.n, 0),
             100.0*SAFE_DIVIDE(COALESCE(ca.n, 0), (SELECT n_people FROM denom WHERE disease='All SARDs'))) AS `All SARDs`,
      FORMAT('%d (%.1f%%)',
             COUNT(DISTINCT IF(rd.disease='RA'      AND rd.race=l.race, rd.person_id, NULL)),
             100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(rd.disease='RA'      AND rd.race=l.race, rd.person_id, NULL)),
                               (SELECT n_people FROM denom WHERE disease='RA'))) AS RA,
      FORMAT('%d (%.1f%%)',
             COUNT(DISTINCT IF(rd.disease='SLE'     AND rd.race=l.race, rd.person_id, NULL)),
             100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(rd.disease='SLE'     AND rd.race=l.race, rd.person_id, NULL)),
                               (SELECT n_people FROM denom WHERE disease='SLE'))) AS SLE,
      FORMAT('%d (%.1f%%)',
             COUNT(DISTINCT IF(rd.disease='SSc'     AND rd.race=l.race, rd.person_id, NULL)),
             100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(rd.disease='SSc'     AND rd.race=l.race, rd.person_id, NULL)),
                               (SELECT n_people FROM denom WHERE disease='SSc'))) AS SSc,
      FORMAT('%d (%.1f%%)',
             COUNT(DISTINCT IF(rd.disease='Sjögren' AND rd.race=l.race, rd.person_id, NULL)),
             100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(rd.disease='Sjögren' AND rd.race=l.race, rd.person_id, NULL)),
                               (SELECT n_people FROM denom WHERE disease='Sjögren'))) AS `Sjögren`,
      FORMAT('%d (%.1f%%)',
             COUNT(DISTINCT IF(rd.disease='IM'      AND rd.race=l.race, rd.person_id, NULL)),
             100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(rd.disease='IM'      AND rd.race=l.race, rd.person_id, NULL)),
                               (SELECT n_people FROM denom WHERE disease='IM'))) AS IM,
      FORMAT('%d (%.1f%%)',
             COUNT(DISTINCT IF(rd.disease='MCTD'    AND rd.race=l.race, rd.person_id, NULL)),
             100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(rd.disease='MCTD'    AND rd.race=l.race, rd.person_id, NULL)),
                               (SELECT n_people FROM denom WHERE disease='MCTD'))) AS MCTD
    FROM levels l
    LEFT JOIN counts_all ca ON ca.race = l.race
    LEFT JOIN race_disease rd ON rd.race = l.race
    GROUP BY l.race, ca.n
    ORDER BY row_label
    """)

    # -----------------------
    # SMOKING (ever smoker)  + shared CTE for pack-years
    # -----------------------

    # Shared smoking CTEs (1 row per person; Yes wins; else No; else NULL)
    COMMON_SMOKE = dedent(f"""
    smoking_raw AS (
      SELECT person_id, answer
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1585857
    ),
    smoking_all AS (
      SELECT
        person_id,
        CASE
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1 ELSE 0 END) = 1 THEN 1
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: No'  THEN 1 ELSE 0 END) = 1 THEN 0
          ELSE NULL
        END AS ever_smoker
      FROM smoking_raw
      GROUP BY person_id
    )
    """)

    sql_row_smoke_ever = dedent(f"""
    {BASE},
    {COMMON_SMOKE},
    smoke_disease AS (
      SELECT i.disease, i.person_id, sa.ever_smoker
      FROM index_by_person i
      LEFT JOIN smoking_all sa USING (person_id)
    ),
    smoke_all AS (
      SELECT a.person_id, sa.ever_smoker
      FROM index_all_by_person a
      LEFT JOIN smoking_all sa USING (person_id)
    ),
    denom AS (
      SELECT disease, COUNT(DISTINCT person_id) AS n_people
      FROM index_by_person
      GROUP BY disease
      UNION ALL
      SELECT 'All SARDs' AS disease, COUNT(DISTINCT person_id) AS n_people
      FROM index_all_by_person
    ),
    n_all AS (
      SELECT COUNTIF(ever_smoker=1) AS n
      FROM smoke_all
    )
    SELECT
      'Ever smoker (n, %)' AS row_label,
      FORMAT('%d (%.1f%%)',
             (SELECT n FROM n_all),
             100.0*SAFE_DIVIDE((SELECT n FROM n_all), (SELECT n_people FROM denom WHERE disease='All SARDs'))) AS `All SARDs`,
      FORMAT('%d (%.1f%%)',
             COUNTIF(disease='RA'      AND ever_smoker=1),
             100.0*SAFE_DIVIDE(COUNTIF(disease='RA'      AND ever_smoker=1), (SELECT n_people FROM denom WHERE disease='RA'))) AS RA,
      FORMAT('%d (%.1f%%)',
             COUNTIF(disease='SLE'     AND ever_smoker=1),
             100.0*SAFE_DIVIDE(COUNTIF(disease='SLE'     AND ever_smoker=1), (SELECT n_people FROM denom WHERE disease='SLE'))) AS SLE,
      FORMAT('%d (%.1f%%)',
             COUNTIF(disease='SSc'     AND ever_smoker=1),
             100.0*SAFE_DIVIDE(COUNTIF(disease='SSc'     AND ever_smoker=1), (SELECT n_people FROM denom WHERE disease='SSc'))) AS SSc,
      FORMAT('%d (%.1f%%)',
             COUNTIF(disease='Sjögren' AND ever_smoker=1),
             100.0*SAFE_DIVIDE(COUNTIF(disease='Sjögren' AND ever_smoker=1), (SELECT n_people FROM denom WHERE disease='Sjögren'))) AS `Sjögren`,
      FORMAT('%d (%.1f%%)',
             COUNTIF(disease='IM'      AND ever_smoker=1),
             100.0*SAFE_DIVIDE(COUNTIF(disease='IM'      AND ever_smoker=1), (SELECT n_people FROM denom WHERE disease='IM'))) AS IM,
      FORMAT('%d (%.1f%%)',
             COUNTIF(disease='MCTD'    AND ever_smoker=1),
             100.0*SAFE_DIVIDE(COUNTIF(disease='MCTD'    AND ever_smoker=1), (SELECT n_people FROM denom WHERE disease='MCTD'))) AS MCTD
    FROM smoke_disease
    """)

    # -----------------------
    # PACK-YEARS
    # -----------------------
    sql_row_packyears = dedent(f"""
    {BASE},
    {COMMON_SMOKE},
    lifestyle AS (
      SELECT person_id, question, answer
      FROM `{CDR}.ds_survey`
      WHERE survey LIKE '%Lifestyle%'
    ),
    avg_daily AS (
      SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS cigs_per_day
      FROM lifestyle
      WHERE question = 'Smoking: Average Daily Cigarette Number'
    ),
    years_smoked AS (
      SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS years
      FROM lifestyle
      WHERE question = 'Smoking: Number Of Years'
    ),
    smoker_ids AS (
      -- semi_join(smoker) equivalent: keep only ever-smokers
      SELECT DISTINCT person_id
      FROM smoking_all
      WHERE ever_smoker = 1
    ),
    packyears_disease AS (
      SELECT i.disease,
             i.person_id,
             (ys.years * (ad.cigs_per_day / 20.0)) AS packyears
      FROM index_by_person i
      JOIN smoker_ids s
        ON s.person_id = i.person_id
      LEFT JOIN avg_daily ad
        ON ad.person_id = i.person_id
      LEFT JOIN years_smoked ys
        ON ys.person_id = i.person_id
    ),
    packyears_all AS (
      SELECT 'All SARDs' AS disease,
             a.person_id,
             (ys.years * (ad.cigs_per_day / 20.0)) AS packyears
      FROM index_all_by_person a
      JOIN smoker_ids s
        ON s.person_id = a.person_id
      LEFT JOIN avg_daily ad
        ON ad.person_id = a.person_id
      LEFT JOIN years_smoked ys
        ON ys.person_id = a.person_id
    )
    SELECT
      'Pack-years (mean, SD)' AS row_label,
      FORMAT('%.1f (%.1f)',
             (SELECT AVG(packyears)         FROM packyears_all WHERE packyears IS NOT NULL),
             (SELECT STDDEV_SAMP(packyears) FROM packyears_all WHERE packyears IS NOT NULL)) AS `All SARDs`,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='RA',      packyears, NULL)), STDDEV_SAMP(IF(disease='RA',      packyears, NULL))) AS RA,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='SLE',     packyears, NULL)), STDDEV_SAMP(IF(disease='SLE',     packyears, NULL))) AS SLE,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='SSc',     packyears, NULL)), STDDEV_SAMP(IF(disease='SSc',     packyears, NULL))) AS SSc,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='Sjögren', packyears, NULL)), STDDEV_SAMP(IF(disease='Sjögren', packyears, NULL))) AS `Sjögren`,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='IM',      packyears, NULL)), STDDEV_SAMP(IF(disease='IM',      packyears, NULL))) AS IM,
      FORMAT('%.1f (%.1f)', AVG(IF(disease='MCTD',    packyears, NULL)), STDDEV_SAMP(IF(disease='MCTD',    packyears, NULL))) AS MCTD
    FROM packyears_disease
    WHERE packyears IS NOT NULL
    """)

    # -----------------------
    # MEDICATIONS (ever use)
    # -----------------------
    sql_med_rows = dedent(f"""
    {BASE},
    denom AS (
      SELECT disease, COUNT(DISTINCT person_id) AS n_people
      FROM index_by_person
      GROUP BY disease
      UNION ALL
      SELECT 'All SARDs' AS disease, COUNT(DISTINCT person_id) AS n_people
      FROM index_all_by_person
    ),
    ALL_MED_IDS AS (
      SELECT DISTINCT drug_concept_id
      FROM {ALL_MED_IDS_UNNEST} AS drug_concept_id
    ),
    std_meds AS (
      SELECT c.concept_id AS std_id
      FROM `{CDR}.concept` c
      JOIN ALL_MED_IDS s ON s.drug_concept_id = c.concept_id
      WHERE c.standard_concept = 'S'
      UNION DISTINCT
      SELECT cr.concept_id_2 AS std_id
      FROM `{CDR}.concept_relationship` cr
      JOIN ALL_MED_IDS s ON s.drug_concept_id = cr.concept_id_1
      WHERE cr.relationship_id = 'Maps to'
    ),
    expanded_meds AS (
      SELECT ca.ancestor_concept_id AS std_id, ca.descendant_concept_id AS drug_concept_id
      FROM `{CDR}.concept_ancestor` ca
      JOIN std_meds sm ON sm.std_id = ca.ancestor_concept_id
      UNION DISTINCT
      SELECT sm.std_id AS std_id, sm.std_id AS drug_concept_id
      FROM std_meds sm
    ),
    med_info AS (
      SELECT sm.std_id AS drug_concept_id, LOWER(c.concept_name) AS medication
      FROM std_meds sm
      LEFT JOIN `{CDR}.concept` c ON c.concept_id = sm.std_id
    ),
    ever_use AS (
      SELECT
        i.disease,
        em.std_id AS drug_concept_id,
        COUNT(DISTINCT i.person_id) AS n
      FROM index_by_person i
      JOIN `{CDR}.drug_exposure` de ON de.person_id = i.person_id
      JOIN expanded_meds em ON em.drug_concept_id = de.drug_concept_id
      GROUP BY i.disease, em.std_id
    ),
    ever_use_all AS (
      SELECT
        'All SARDs' AS disease,
        em.std_id AS drug_concept_id,
        COUNT(DISTINCT a.person_id) AS n
      FROM index_all_by_person a
      JOIN `{CDR}.drug_exposure` de ON de.person_id = a.person_id
      JOIN expanded_meds em ON em.drug_concept_id = de.drug_concept_id
      GROUP BY em.std_id
    ),
    eu AS (
      SELECT * FROM ever_use
      UNION ALL
      SELECT * FROM ever_use_all
    )
    SELECT
      mi.medication AS row_label,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='All SARDs', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='All SARDs', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='All SARDs'))) AS `All SARDs`,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='RA', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='RA', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='RA'))) AS RA,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='SLE', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='SLE', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='SLE'))) AS SLE,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='SSc', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='SSc', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='SSc'))) AS SSc,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='Sjögren', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='Sjögren', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='Sjögren'))) AS `Sjögren`,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='IM', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='IM', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='IM'))) AS IM,
      FORMAT('%d (%.1f%%)',
             SUM(IF(eu.disease='MCTD', eu.n, 0)),
             100.0*SAFE_DIVIDE(SUM(IF(eu.disease='MCTD', eu.n, 0)),
                               (SELECT n_people FROM denom WHERE disease='MCTD'))) AS MCTD
    FROM med_info mi
    JOIN eu ON eu.drug_concept_id = mi.drug_concept_id
    GROUP BY mi.medication
    """)

    # -----------------------
    # ILD rows (loose + strict)
    # -----------------------
    blocks_sql = [
        sql_row_age,
        sql_row_female,
        sql_row_male,
        sql_row_race_all,
        sql_row_smoke_ever,
        sql_row_packyears,
        sql_med_rows,
    ]

    if include_ild_rows:
        # Loose: avoid self-join by using min date and "exists later" pattern
        sql_ild_relaxed = dedent(f"""
        {BASE},
        ild_events AS (
          SELECT person_id, DATE(entry_date) AS dt
          FROM `{CDR}.cb_search_all_events`
          WHERE concept_id IN {ILD_UNNEST}
            AND is_standard = 0
        ),
        first_ild AS (
          SELECT person_id, MIN(dt) AS first_dt
          FROM ild_events
          GROUP BY person_id
        ),
        ild_relaxed_raw AS (
          SELECT f.person_id
          FROM first_ild f
          WHERE EXISTS (
            SELECT 1
            FROM ild_events e
            WHERE e.person_id = f.person_id
              AND e.dt >= DATE_ADD(f.first_dt, INTERVAL 30 DAY)
          )
        ),
        ild_relaxed_in_cohort AS (
          SELECT DISTINCT r.person_id
          FROM ild_relaxed_raw r
          JOIN index_all_by_person a USING (person_id)
        )
        SELECT
          'Loose definition' AS row_label,
          FORMAT('%d (%.1f%%)',
                 (SELECT COUNT(*) FROM ild_relaxed_in_cohort),
                 100.0*SAFE_DIVIDE((SELECT COUNT(*) FROM ild_relaxed_in_cohort),
                                   (SELECT COUNT(*) FROM index_all_by_person))) AS `All SARDs`,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='RA', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='RA', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='RA'))) AS RA,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='SLE', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='SLE', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='SLE'))) AS SLE,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='SSc', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='SSc', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='SSc'))) AS SSc,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='Sjögren', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='Sjögren', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='Sjögren'))) AS `Sjögren`,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='IM', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='IM', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='IM'))) AS IM,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='MCTD', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='MCTD', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='MCTD'))) AS MCTD
        FROM ild_relaxed_in_cohort r
        LEFT JOIN index_by_person i USING (person_id)
        """)

        # Strict: your original approach is okay; keep it but reuse BASE and avoid repeated cohort rebuild
        sql_ild_stringent = dedent(f"""
        {BASE},
        pft_std_seeds AS (
          SELECT concept_id AS std_id
          FROM `{CDR}.concept`
          WHERE standard_concept='S'
            AND (LOWER(concept_name) LIKE '%pulmonary function%'
                 OR LOWER(concept_name) LIKE '%spirometr%')
        ),
        pft_descendants AS (
          SELECT ca.descendant_concept_id AS cid
          FROM `{CDR}.concept_ancestor` ca
          JOIN pft_std_seeds s ON s.std_id = ca.ancestor_concept_id
          UNION DISTINCT
          SELECT std_id AS cid FROM pft_std_seeds
        ),
        pft_events AS (
          SELECT person_id, DATE(procedure_date) AS dt
          FROM `{CDR}.procedure_occurrence`
          WHERE procedure_concept_id IN (SELECT cid FROM pft_descendants)
          UNION ALL
          SELECT person_id, DATE(measurement_date) AS dt
          FROM `{CDR}.measurement`
          WHERE measurement_concept_id IN (SELECT cid FROM pft_descendants)
        ),
        first_ild_date AS (
          SELECT person_id, MIN(DATE(condition_start_datetime)) AS ild_date
          FROM `{CDR}.condition_occurrence`
          WHERE condition_source_concept_id IN {ILD_UNNEST}
          GROUP BY person_id
        ),
        ild_stringent_raw AS (
          SELECT DISTINCT f.person_id
          FROM first_ild_date f
          JOIN pft_events pe
            ON pe.person_id = f.person_id
           AND pe.dt BETWEEN DATE_SUB(f.ild_date, INTERVAL 180 DAY)
                         AND DATE_SUB(f.ild_date, INTERVAL 7 DAY)
        ),
        ild_stringent_in_cohort AS (
          SELECT DISTINCT s.person_id
          FROM ild_stringent_raw s
          JOIN index_all_by_person a USING (person_id)
        )
        SELECT
          'Strict definition' AS row_label,
          FORMAT('%d (%.1f%%)',
                 (SELECT COUNT(*) FROM ild_stringent_in_cohort),
                 100.0*SAFE_DIVIDE((SELECT COUNT(*) FROM ild_stringent_in_cohort),
                                   (SELECT COUNT(*) FROM index_all_by_person))) AS `All SARDs`,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='RA', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='RA', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='RA'))) AS RA,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='SLE', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='SLE', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='SLE'))) AS SLE,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='SSc', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='SSc', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='SSc'))) AS SSc,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='Sjögren', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='Sjögren', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='Sjögren'))) AS `Sjögren`,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='IM', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='IM', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='IM'))) AS IM,
          FORMAT('%d (%.1f%%)',
                 COUNT(DISTINCT IF(i.disease='MCTD', i.person_id, NULL)),
                 100.0*SAFE_DIVIDE(COUNT(DISTINCT IF(i.disease='MCTD', i.person_id, NULL)),
                                   (SELECT COUNT(DISTINCT person_id) FROM index_by_person WHERE disease='MCTD'))) AS MCTD
        FROM ild_stringent_in_cohort s
        LEFT JOIN index_by_person i USING (person_id)
        """)

        blocks_sql.extend([sql_ild_relaxed, sql_ild_stringent])

    # -----------------------
    # Execute blocks (with debug)
    # -----------------------
    blocks = []
    for k, s in enumerate(blocks_sql, start=1):
        if debug:
            print(f"Running block {k}/{len(blocks_sql)}...")
        blocks.append(_read(s))

    results_table = pd.concat(blocks, ignore_index=True)

    # Enforce column order
    cols = ["row_label", "All SARDs", "RA", "SLE", "SSc", "Sjögren", "IM", "MCTD"]
    results_table = results_table[cols]

    # Turn dynamic race labels from "RACE: <label>" into plain labels
    race_mask = results_table["row_label"].astype(str).str.startswith("RACE: ", na=False)
    race_labels_raw = sorted(results_table.loc[race_mask, "row_label"].unique())
    race_display_labels = [lbl.replace("RACE: ", "") for lbl in race_labels_raw]
    results_table.loc[race_mask, "row_label"] = results_table.loc[race_mask, "row_label"].str.replace(
        r"^RACE:\s*", "", regex=True
    )

    # Definition row
    def_row = {
        "row_label": "Definition",
        "All SARDs": definition_row["All SARDs"],
        "RA":        definition_row["RA"],
        "SLE":       definition_row["SLE"],
        "SSc":       definition_row["SSc"],
        "Sjögren":   definition_row["Sjögren"],
        "IM":        definition_row["IM"],
        "MCTD":      definition_row["MCTD"],
    }

    # Section headers
    header_rows = [
        {"row_label": "Self-reported Race", "All SARDs": "", "RA": "", "SLE": "", "SSc": "", "Sjögren": "", "IM": "", "MCTD": ""},
        {"row_label": "Immunosuppressive medication use", "All SARDs": "", "RA": "", "SLE": "", "SSc": "", "Sjögren": "", "IM": "", "MCTD": ""},
        {"row_label": "Interstitial Lung Disease", "All SARDs": "", "RA": "", "SLE": "", "SSc": "", "Sjögren": "", "IM": "", "MCTD": ""},
    ]

    results_table = pd.concat(
        [pd.DataFrame([def_row]), results_table, pd.DataFrame(header_rows)],
        ignore_index=True
    )

    med_rows_order = [
        "abatacept",
        "adalimumab",
        "anakinra",
        "anifrolumab",
        "azathioprine",
        "cyclophosphamide",
        "cyclosporine",
        "etanercept",
        "gold",
        "hydroxychloroquine",
        "immunoglobulin g",   # <-- FIXED (was "immunoglobulin G")
        "infliximab",
        "leflunomide",
        "methotrexate",
        "minocycline",
        "mycophenolate",
        "mycophenolate mofetil",
        "penicillamine",
        "rituximab",
        "sulfasalazine",
    ]

    default_row_order = [
        "Definition",
        "Age (years, mean, SD)",
        "Female sex (n, %)",
        "Male sex (n, %)",
        "Self-reported Race",
    ] + race_display_labels + [
        "Ever smoker (n, %)",
        "Pack-years (mean, SD)",
        "Immunosuppressive medication use",
    ] + med_rows_order + [
        "Interstitial Lung Disease",
        "Loose definition",
        "Strict definition",
    ]

    if row_order is None:
        row_order_effective = default_row_order
    else:
        row_order_effective = list(row_order)

    if not include_ild_rows:
        row_order_effective = [
            r for r in row_order_effective
            if r not in ("Interstitial Lung Disease", "Loose definition", "Strict definition")
        ]

    results_table = results_table[results_table["row_label"].isin(row_order_effective)]
    results_table["row_label"] = pd.Categorical(
        results_table["row_label"],
        categories=row_order_effective,
        ordered=True,
    )
    results_table = results_table.sort_values("row_label").reset_index(drop=True)

    # Cohort sizes for suppression and headers
    _totals_sql = dedent(f"""
    {BASE}
    SELECT disease, COUNT(DISTINCT person_id) AS n_people
    FROM index_by_person
    GROUP BY disease
    UNION ALL
    SELECT 'All SARDs' AS disease, COUNT(DISTINCT person_id) AS n_people
    FROM index_all_by_person
    """)
    _totals_df = _read(_totals_sql)
    _cohort_n = dict(zip(_totals_df["disease"], _totals_df["n_people"]))

    _num_pct_re = re.compile(r"^\s*(\d+)\s*\(\s*([0-9.]+)\s*%\s*\)\s*$")
    _disease_cols = ["All SARDs", "RA", "SLE", "SSc", "Sjögren", "IM", "MCTD"]

    _suppress_labels = set(
        ["Ever smoker (n, %)", "Female sex (n, %)", "Male sex (n, %)"]
        + med_rows_order
        + race_display_labels
    )
    if include_ild_rows:
        _suppress_labels.update(["Loose definition", "Strict definition"])

    _suppress_rows = results_table["row_label"].isin(_suppress_labels)

    for idx in results_table.index[_suppress_rows]:
        for col in _disease_cols:
            txt = results_table.at[idx, col]
            if isinstance(txt, str):
                m = _num_pct_re.match(txt)
                if m:
                    n_val = int(m.group(1))
                    if n_val < 20:
                        denom = _cohort_n.get(col)
                        if denom and denom > 0:
                            pct20 = 100.0 * 20.0 / float(denom)
                            results_table.at[idx, col] = f"< 20 (< {pct20:.1f}%)"
                        else:
                            results_table.at[idx, col] = "< 20"

    # Rename row_label and add counts to column headers
    results_table = results_table.rename(columns={"row_label": "Characteristic"})

    col_label_map = {}
    for d in _disease_cols:
        if d in results_table.columns and d in _cohort_n:
            col_label_map[d] = f"{d} (n={_cohort_n[d]})"
    results_table = results_table.rename(columns=col_label_map)

    return results_table


NameError: name 'pd' is not defined

In [3]:
# Build Table 1 (All SARD cohort)

# Definitions for the main SARD cohort
sard_definition_row = {
    "All SARDs": "All participants meeting any RA, SLE, SSc, Sjögren, IM, or MCTD algorithm",
    "RA":        "2+ RA ICD-9/10 codes (714.x, M05.x, M06.x), 7+ days apart, and 1+ RA medication on/within 365 days after any RA code",
    "SLE":       "2+ SLE ICD-9/10 codes (710.0 or M32.x), 30+ days apart, AND 1+ SLE medication on/after any SLE code",
    "SSc":       "1+ inpatient OR 2+ outpatient SSc codes (710.1, 517.2, M34.x), 30+ days apart",
    "Sjögren":   "2+ Sjögren’s codes (710.2, M35.0), 28+ days apart",
    "IM":        "1+ inpatient OR 2+ outpatient IIM codes (710.3, 710.4, 728.81, M33.x, M60.10), 30-365 days apart, AND 1+ IIM medication on/within 365 days of any IIM code",
    "MCTD":      "1+ inpatient OR 2+ outpatient MCTD codes (710.8, M35.1), 30-365 days apart",
}

# For Table 1 (all SARDs), cohort_sql is just final_sql and we include ILD rows
results_table = build_sard_summary_table(
    cohort_sql=final_sql_materialized,
    definition_row=sard_definition_row,
    include_ild_rows=True,
    debug=True,
)

# Canonical row order for all downstream summary tables (ILD, no-ILD, etc.)
canonical_row_order = results_table["Characteristic"].tolist()

results_table

NameError: name 'build_sard_summary_table' is not defined

In [6]:
# Export to CSV
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path   = f"sard_summary_{timestamp}.csv"

results_table.to_csv(csv_path, index=False)
print(f"CSV written: {csv_path}")


NameError: name 'datetime' is not defined

In [16]:
# ILD-relaxed SARD cohort SQL

ILD_RELAXED_PEOPLE_SQL = dedent(f"""
WITH ild_relaxed_raw AS (
  SELECT DISTINCT t1.person_id
  FROM (
    SELECT person_id, entry_date
    FROM `{CDR}.cb_search_all_events`
    WHERE concept_id IN {ILD_UNNEST}
      AND is_standard = 0
  ) t1
  JOIN (
    SELECT person_id, entry_date
    FROM `{CDR}.cb_search_all_events`
    WHERE concept_id IN {ILD_UNNEST}
      AND is_standard = 0
  ) t2
    ON t1.person_id = t2.person_id
   AND t2.entry_date >= DATE_ADD(t1.entry_date, INTERVAL 30 DAY)
)
SELECT person_id
FROM ild_relaxed_raw
""")

ild_relaxed_cohort_sql = dedent(f"""
SELECT *
FROM ({final_sql_materialized}) c
WHERE c.person_id IN (
  {ILD_RELAXED_PEOPLE_SQL}
)
""")


In [17]:
# ILD-stringent SARD cohort SQL

ILD_STRINGENT_PEOPLE_SQL = dedent(f"""
WITH
pft_std_seeds AS (
  SELECT concept_id AS std_id
  FROM `{CDR}.concept`
  WHERE standard_concept = 'S'
    AND (LOWER(concept_name) LIKE '%pulmonary function%'
         OR LOWER(concept_name) LIKE '%spirometr%')
),
pft_descendants AS (
  SELECT ca.descendant_concept_id AS cid
  FROM `{CDR}.concept_ancestor` ca
  JOIN pft_std_seeds s
    ON s.std_id = ca.ancestor_concept_id
  UNION DISTINCT
  SELECT std_id AS cid
  FROM pft_std_seeds
),
pft_events AS (
  SELECT person_id, DATE(procedure_date) AS dt
  FROM `{CDR}.procedure_occurrence`
  WHERE procedure_concept_id IN (SELECT cid FROM pft_descendants)
  UNION ALL
  SELECT person_id, DATE(measurement_date) AS dt
  FROM `{CDR}.measurement`
  WHERE measurement_concept_id IN (SELECT cid FROM pft_descendants)
),
first_ild_date AS (
  SELECT person_id, MIN(DATE(condition_start_datetime)) AS ild_date
  FROM `{CDR}.condition_occurrence`
  WHERE condition_source_concept_id IN {ILD_UNNEST}
  GROUP BY person_id
),
ild_stringent_raw AS (
  SELECT DISTINCT f.person_id
  FROM first_ild_date f
  JOIN pft_events pe
    ON pe.person_id = f.person_id
   AND pe.dt BETWEEN DATE_SUB(f.ild_date, INTERVAL 180 DAY)
                 AND DATE_SUB(f.ild_date, INTERVAL 7 DAY)
)
SELECT person_id
FROM ild_stringent_raw
""")

ild_stringent_cohort_sql = dedent(f"""
SELECT *
FROM ({final_sql_materialized}) c
WHERE c.person_id IN (
  {ILD_STRINGENT_PEOPLE_SQL}
)
""")


In [18]:
# Definition rows for ILD tables

# "Definition" row texts for the ILD-specific tables.
# The keys must match the disease columns.
ild_relaxed_definition_row = {
    "All SARDs": "SARD cohort members meeting loose ILD definition (2+ ILD codes ≥30 days apart)",
    "RA":        "RA participants with loose ILD definition",
    "SLE":       "SLE participants with loose ILD definition",
    "SSc":       "SSc participants with loose ILD definition",
    "Sjögren":   "Sjögren participants with loose ILD definition",
    "IM":        "IM participants with loose ILD definition",
    "MCTD":      "MCTD participants with loose ILD definition",
}

ild_stringent_definition_row = {
    "All SARDs": "SARD cohort members meeting strict ILD definition (PFT 7–180 days before first ILD code)",
    "RA":        "RA participants with strict ILD definition",
    "SLE":       "SLE participants with strict ILD definition",
    "SSc":       "SSc participants with strict ILD definition",
    "Sjögren":   "Sjögren participants with strict ILD definition",
    "IM":        "IM participants with strict ILD definition",
    "MCTD":      "MCTD participants with strict ILD definition",
}


# Build loose table
results_table_ild_loose = build_sard_summary_table(
    cohort_sql=ild_relaxed_cohort_sql,
    definition_row=ild_relaxed_definition_row,
    include_ild_rows=False,
    row_order=canonical_row_order,
)

results_table_ild_loose


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


,Characteristic,All SARDs (n=710),RA (n=355),SLE (n=136),SSc (n=167),Sjögren (n=224),IM (n=65),MCTD (n=85)
0,Definition,SARD cohort members meeting loose ILD definition (2+ ILD codes ≥30 days apart),RA participants with loose ILD definition,SLE participants with loose ILD definition,SSc participants with loose ILD definition,Sjögren participants with loose ILD definition,IM participants with loose ILD definition,MCTD participants with loose ILD definition
1,"Age (years, mean, SD)",54.2 (13.0),53.9 (12.9),47.4 (12.2),52.2 (12.4),57.9 (12.5),51.2 (13.1),49.7 (13.1)
2,"Female sex (n, %)",582 (82.0%),285 (80.3%),127 (93.4%),143 (85.6%),198 (88.4%),46 (70.8%),74 (87.1%)
3,"Male sex (n, %)",120 (16.9%),66 (18.6%),< 20 (< 14.7%),21 (12.6%),22 (9.8%),< 20 (< 30.8%),< 20 (< 23.5%)
4,Self-reported Race,,,,,,,
5,Another single population,< 20 (< 2.8%),< 20 (< 5.6%),< 20 (< 14.7%),< 20 (< 12.0%),< 20 (< 8.9%),< 20 (< 30.8%),< 20 (< 23.5%)
6,More than one population,28 (3.9%),< 20 (< 5.6%),< 20 (< 14.7%),< 20 (< 12.0%),< 20 (< 8.9%),< 20 (< 30.8%),< 20 (< 23.5%)
7,PMI: Prefer Not To Answer,< 20 (< 2.8%),< 20 (< 5.6%),< 20 (< 14.7%),< 20 (< 12.0%),< 20 (< 8.9%),< 20 (< 30.8%),< 20 (< 23.5%)
8,PMI: Skip,< 20 (< 2.8%),< 20 (< 5.6%),< 20 (< 14.7%),< 20 (< 12.0%),< 20 (< 8.9%),< 20 (< 30.8%),< 20 (< 23.5%)
9,What Race Ethnicity: Asian,< 20 (< 2.8%),< 20 (< 5.6%),< 20 (< 14.7%),< 20 (< 12.0%),< 20 (< 8.9%),< 20 (< 30.8%),< 20 (< 23.5%)


In [7]:
# Strict Table
results_table_ild_strict = build_sard_summary_table(
    cohort_sql=ild_stringent_cohort_sql,
    definition_row=ild_stringent_definition_row,
    include_ild_rows=False,
    row_order=canonical_row_order,
)

results_table_ild_strict


NameError: name 'build_sard_summary_table' is not defined

In [20]:
# Export both to CSV
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

csv_loose = f"sard_ild_loose_summary_{timestamp}.csv"
csv_strict = f"sard_ild_stringent_summary_{timestamp}.csv"

results_table_ild_loose.to_csv(csv_loose, index=False)
results_table_ild_strict.to_csv(csv_strict, index=False)

print(f"CSV written: {csv_loose}")
print(f"CSV written: {csv_strict}")


CSV written: sard_ild_loose_summary_20260327-124325.csv
CSV written: sard_ild_stringent_summary_20260327-124325.csv


In [21]:
# Build SARD-noILD cohort SQL (no known ILD codes ever)

NO_ILD_SQL = dedent(f"""
WITH any_ild AS (
  SELECT DISTINCT person_id
  FROM `{CDR}.cb_search_all_events`
  WHERE concept_id IN {ILD_UNNEST}
)
SELECT * FROM ({final_sql_materialized}) c
WHERE c.person_id NOT IN (SELECT person_id FROM any_ild)
""")

# Labels for the “Definition” row
no_ild_definition_row = {
    "All SARDs": "SARD cohort members with NO known ILD codes",
    "RA":        "RA participants with NO ILD codes",
    "SLE":       "SLE participants with NO ILD codes",
    "SSc":       "SSc participants with NO ILD codes",
    "Sjögren":   "Sjögren participants with NO ILD codes",
    "IM":        "IM participants with NO ILD codes",
    "MCTD":      "MCTD participants with NO ILD codes",
}

# Use existing summary tables and canonical row order

# results_table: full SARD summary (built earlier)
# results_table_ild_loose: SARD-ILD (loose) summary (built earlier)
# canonical_row_order: from the full SARD table
#   canonical_row_order = results_table["Characteristic"].tolist()

table_overall = results_table
table_ILD = results_table_ild_loose

# Build SARD-noILD summary, reusing the overall-cohort row order
table_noILD = build_sard_summary_table(
    cohort_sql=NO_ILD_SQL,
    definition_row=no_ild_definition_row,
    include_ild_rows=False,
    row_order=canonical_row_order,
)

# Align ILD and no-ILD tables on common characteristics
rows_ILD = table_ILD["Characteristic"].tolist()
rows_noILD = set(table_noILD["Characteristic"].tolist())
rows = [r for r in rows_ILD if r in rows_noILD]

table_ILD_sub = table_ILD[table_ILD["Characteristic"].isin(rows)].reset_index(drop=True)
table_noILD_sub = table_noILD[table_noILD["Characteristic"].isin(rows)].reset_index(drop=True)

# Extract Ns from column headers
_n_header_re = re.compile(r"n=(\d+)")
col_ILD = table_ILD_sub.columns[1]
col_noILD = table_noILD_sub.columns[1]

m1 = _n_header_re.search(col_ILD)
m2 = _n_header_re.search(col_noILD)
if not (m1 and m2):
    raise ValueError("Could not extract group sizes from column headers.")

n_ILD = int(m1.group(1))
n_noILD = int(m2.group(1))

# Build the Table 3 scaffold using aligned tables
table3 = pd.DataFrame({
    "Characteristic": rows,
    col_ILD: table_ILD_sub[col_ILD].values,
    col_noILD: table_noILD_sub[col_noILD].values,
    "p-value": "",
})

# Helper regex to parse “N (X.X%)”
_num_pct_re = re.compile(r"^\s*(\d+)\s*\(\s*([0-9.]+)%\s*\)\s*$")

def _parse_n(cell):
    if not isinstance(cell, str):
        return None
    m = _num_pct_re.match(cell)
    return int(m.group(1)) if m else None

# Pack-years Wilcoxon rank-sum: individual-level data

PACKYEARS_SQL_TEMPLATE = """
WITH
cohort AS (
  {cohort_sql}
),
index_all_by_person AS (
  SELECT person_id, MIN(DATE(condition_start_datetime)) AS index_date
  FROM cohort
  GROUP BY person_id
),
smoking_raw AS (
  SELECT person_id, answer
  FROM `{CDR}.ds_survey`
  WHERE question_concept_id = 1585857
),
smoking_map AS (
  SELECT person_id,
         CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1
              WHEN answer='100 Cigs Lifetime: No'  THEN 0
              ELSE NULL END AS ever_smoker
  FROM smoking_raw
),
smoking_all AS (
  SELECT 'All SARDs' AS disease, a.person_id, sm.ever_smoker
  FROM index_all_by_person a
  LEFT JOIN smoking_map sm ON sm.person_id = a.person_id
),
lifestyle AS (
  SELECT person_id, question, answer
  FROM `{CDR}.ds_survey`
  WHERE survey LIKE '%Lifestyle%'
),
avg_daily AS (
  SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS cigs_per_day
  FROM lifestyle
  WHERE question = 'Smoking: Average Daily Cigarette Number'
),
years_smoked AS (
  SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS years
  FROM lifestyle
  WHERE question = 'Smoking: Number Of Years'
),
packyears_all AS (
  SELECT 'All SARDs' AS disease,
         a.person_id,
         (ys.years * (ad.cigs_per_day / 20.0)) AS packyears
  FROM index_all_by_person a
  LEFT JOIN avg_daily ad    ON ad.person_id = a.person_id
  LEFT JOIN years_smoked ys ON ys.person_id = a.person_id
)
SELECT packyears
FROM packyears_all
WHERE packyears IS NOT NULL
"""

# ILD+ and noILD cohort SQLs were defined earlier in the notebook
pack_ILD_sql = dedent(PACKYEARS_SQL_TEMPLATE.format(
    cohort_sql=ild_relaxed_cohort_sql,
    CDR=CDR,
))
pack_noILD_sql = dedent(PACKYEARS_SQL_TEMPLATE.format(
    cohort_sql=NO_ILD_SQL,
    CDR=CDR,
))

pack_ILD_df = pandas_gbq.read_gbq(pack_ILD_sql, dialect="standard")
pack_noILD_df = pandas_gbq.read_gbq(pack_noILD_sql, dialect="standard")

pack_ILD = pack_ILD_df["packyears"].dropna().values
pack_noILD = pack_noILD_df["packyears"].dropna().values

# Compute p-values

pvals = {}

for row in rows:
    # Skip purely descriptive header rows
    if row in ("Definition", "Self-reported Race", "Immunosuppressive medication use"):
        continue

    cell_ILD = table_ILD_sub.loc[table_ILD_sub["Characteristic"] == row, col_ILD].iloc[0]
    cell_noILD = table_noILD_sub.loc[table_noILD_sub["Characteristic"] == row, col_noILD].iloc[0]

    # Age: t-test using summary stats
    if row.startswith("Age"):
        def _extract_mean_sd(x):
            try:
                m = re.findall(r"[0-9.]+", x)
                return float(m[0]), float(m[1])
            except Exception:
                return None

        m1sd1 = _extract_mean_sd(cell_ILD)
        m2sd2 = _extract_mean_sd(cell_noILD)

        if m1sd1 and m2sd2:
            (m1, s1), (m2, s2) = m1sd1, m2sd2
            se = np.sqrt((s1**2)/n_ILD + (s2**2)/n_noILD)
            if se == 0:
                pvals[row] = ""
            else:
                t = (m1 - m2) / se
                df = n_ILD + n_noILD - 2
                p = 2 * (1 - stats.t.cdf(abs(t), df))
                pvals[row] = p
        continue

    # Pack-years: Wilcoxon rank-sum (Mann–Whitney U) using individual-level data
    if row.startswith("Pack-years"):
        if len(pack_ILD) > 0 and len(pack_noILD) > 0:
            stat, p = stats.mannwhitneyu(pack_ILD, pack_noILD, alternative="two-sided")
            pvals[row] = p
        continue

    # Everything else: chi-square based on counts in "N (X.X%)"
    yes_ILD = _parse_n(cell_ILD)
    yes_noILD = _parse_n(cell_noILD)

    # If parsing fails or AoU suppression (<20): no p-value
    if yes_ILD is None or yes_noILD is None:
        continue

    no_ILD_yes = n_ILD - yes_ILD
    no_noILD_yes = n_noILD - yes_noILD

    if min([yes_ILD, no_ILD_yes, yes_noILD, no_noILD_yes]) < 20:
        continue  # enforce AoU suppression rule

    chi_table = [[yes_ILD, no_ILD_yes],
                 [yes_noILD, no_noILD_yes]]

    chi2, p, _, _ = stats.chi2_contingency(chi_table)
    pvals[row] = p

# Assign p-values into table3
def _format_p(p):
    if p == "" or p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    return f"{p:.4f}"

table3["p-value"] = table3["Characteristic"].map(lambda r: _format_p(pvals.get(r, "")))

# Final Table 3
table3


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


,Characteristic,All SARDs (n=710),All SARDs (n=8478),p-value
0,Definition,SARD cohort members meeting loose ILD definition (2+ ILD codes ≥30 days apart),SARD cohort members with NO known ILD codes,
1,"Age (years, mean, SD)",54.2 (13.0),51.6 (14.7),0.0000
2,"Female sex (n, %)",582 (82.0%),6982 (82.4%),0.8372
3,"Male sex (n, %)",120 (16.9%),1408 (16.6%),0.8812
4,Self-reported Race,,,
5,Another single population,< 20 (< 2.8%),126 (1.5%),
6,More than one population,28 (3.9%),380 (4.5%),0.5658
7,PMI: Prefer Not To Answer,< 20 (< 2.8%),45 (0.5%),
8,PMI: Skip,< 20 (< 2.8%),120 (1.4%),
9,What Race Ethnicity: Asian,< 20 (< 2.8%),187 (2.2%),


In [38]:
# Build Table S2: SARD-noILD cohort summary

table_S2 = build_sard_summary_table(
    cohort_sql=NO_ILD_SQL,
    definition_row=no_ild_definition_row,
    include_ild_rows=False,
    row_order=canonical_row_order,
    debug=True,   # switch to False once it runs cleanly
)

table_S2

NameError: name 'build_sard_summary_table' is not defined

In [37]:
# Export to CSV

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = f"ild_no_ild_{timestamp}.csv"

table3.to_csv(csv_path, index=False)
print(f"CSV written: {csv_path}")


NameError: name 'datetime' is not defined

## 3. Logistic Regression
The following cells perform anadjusted and adjusted logistic regression to assess whether age, sex, and smoking are associated with SARD-ILD.

In [36]:
# Logistic regression for ILD Status (NO pack-years)
def build_table1_person_df_sql(final_sql: str) -> str:
    """
    Builds ONE person-level dataset with:
      - disease (and All SARDs)
      - ild_status (1=ILD relaxed/loose, 0=noILD ever)
      - age_years at index_date (float)
      - male (1/0)
      - ever_smoker (1/0)

    Reuses:
      - final_sql (disease cohort logic)
      - ILD_UNNEST, CDR
    """
    return dedent(f"""
    WITH
    cohort AS (
      {final_sql}
    ),

    -- Per-disease and all-SARD index dates
    index_by_person AS (
      SELECT disease, person_id, MIN(DATE(condition_start_datetime)) AS index_date
      FROM cohort
      GROUP BY disease, person_id
    ),
    index_all AS (
      SELECT person_id, MIN(DATE(condition_start_datetime)) AS index_date
      FROM cohort
      GROUP BY person_id
    ),

    -- ILD cases: 2+ ILD concept_ids at least 30 days apart (relaxed/loose)
    ild_cases AS (
      SELECT DISTINCT t1.person_id
      FROM `{CDR}.cb_search_all_events` t1
      JOIN `{CDR}.cb_search_all_events` t2
        ON t1.person_id = t2.person_id
       AND DATE(t2.entry_date) >= DATE_ADD(DATE(t1.entry_date), INTERVAL 30 DAY)
      WHERE t1.concept_id IN {ILD_UNNEST}
        AND t2.concept_id IN {ILD_UNNEST}
        AND t1.is_standard = 0
        AND t2.is_standard = 0
    ),

    -- "Any ILD ever" (controls must have none of these, per your NO_ILD logic)
    any_ild AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {ILD_UNNEST}
    ),
    no_ild_controls AS (
      SELECT DISTINCT c.person_id
      FROM cohort c
      WHERE c.person_id NOT IN (SELECT person_id FROM any_ild)
    ),

    -- Persons (cases OR controls), within each disease stratum
    disease_rows AS (
      SELECT
        i.disease,
        i.person_id,
        i.index_date,
        CASE
          WHEN i.person_id IN (SELECT person_id FROM ild_cases) THEN 1
          WHEN i.person_id IN (SELECT person_id FROM no_ild_controls) THEN 0
          ELSE NULL
        END AS ild_status
      FROM index_by_person i
      WHERE i.disease IN ('RA','SLE','SSc','Sjögren','IM','MCTD')
    ),
    all_sard_rows AS (
      SELECT
        'All SARDs' AS disease,
        a.person_id,
        a.index_date,
        CASE
          WHEN a.person_id IN (SELECT person_id FROM ild_cases) THEN 1
          WHEN a.person_id IN (SELECT person_id FROM no_ild_controls) THEN 0
          ELSE NULL
        END AS ild_status
      FROM index_all a
    ),
    rows_union AS (
      SELECT * FROM disease_rows
      UNION ALL
      SELECT * FROM all_sard_rows
    ),

    -- Demographics
    p AS (
      SELECT person_id, birth_datetime, sex_at_birth_concept_id
      FROM `{CDR}.person`
    ),
    sex_map AS (
      SELECT concept_id, concept_name
      FROM `{CDR}.concept`
    ),

    -- Ever smoker (one row per person; "ever yes" wins; else "ever no"; else NULL)
    smoking_map AS (
      SELECT
        person_id,
        CASE
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1 ELSE 0 END) = 1 THEN 1
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: No'  THEN 1 ELSE 0 END) = 1 THEN 0
          ELSE NULL
        END AS ever_smoker
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1585857
      GROUP BY person_id
    )

    SELECT
      r.disease,
      r.person_id,
      r.ild_status,

      -- float age in years at index date
      SAFE_DIVIDE(DATE_DIFF(r.index_date, DATE(p.birth_datetime), DAY), 365.25) AS age_years,

      CASE
        WHEN LOWER(cm.concept_name) = 'male'   THEN 1
        WHEN LOWER(cm.concept_name) = 'female' THEN 0
        ELSE NULL
      END AS male,

      sm.ever_smoker
    FROM rows_union r
    JOIN p USING (person_id)
    LEFT JOIN sex_map cm ON cm.concept_id = p.sex_at_birth_concept_id
    LEFT JOIN smoking_map sm USING (person_id)
    WHERE r.ild_status IS NOT NULL
    """)

# IMPORTANT: pass the SAME SQL string you used before (likely final_sql_materialized)
PERSON_LEVEL_SQL = build_table1_person_df_sql(final_sql_materialized)

person_df = pandas_gbq.read_gbq(PERSON_LEVEL_SQL, dialect="standard")
print("Person-level rows:", len(person_df))
person_df.head()


NameError: name 'final_sql_materialized' is not defined

In [35]:
# Table labels
DISPLAY = {"RA":"RA", "SSc":"SSc", "Sjögren":"SjD", "SLE":"SLE", "MCTD":"MCTD", "IM":"IIM", "All SARDs":"SARD"}

def _fmt_or_ci(or_val, lo, hi):
    if or_val is None or lo is None or hi is None:
        return ""
    return f"{or_val:.2f} ({lo:.2f}, {hi:.2f})"

def _fit_logit(df, y, Xcols):
    d = df[[y] + Xcols].dropna().copy()
    if d.empty or d[y].nunique() < 2:
        return None
    X = sm.add_constant(d[Xcols].astype(float), has_constant="add")
    yv = d[y].astype(float)
    try:
        return sm.Logit(yv, X).fit(disp=0)
    except Exception:
        return None

def _or_ci(model, term):
    try:
        beta = model.params[term]
        lo, hi = model.conf_int().loc[term].tolist()
        return float(np.exp(beta)), float(np.exp(lo)), float(np.exp(hi))
    except Exception:
        return None, None, None

def _unadj_adj(df, exposure, adj_covars, per10=False):
    d = df.copy()
    if per10:
        d[exposure] = d[exposure] / 10.0

    m1 = _fit_logit(d, "ild_status", [exposure])
    unadj = _fmt_or_ci(*_or_ci(m1, exposure)) if m1 is not None else ""

    # adjusted model includes exposure + the other covariates (so term exists)
    covars = [c for c in adj_covars if c != exposure]
    m2 = _fit_logit(d, "ild_status", [exposure] + covars)
    adj = _fmt_or_ci(*_or_ci(m2, exposure)) if m2 is not None else ""

    return unadj, adj

def build_table1_shell(person_df, min_group_n=20):
    strata = ["All SARDs", "RA", "SSc", "Sjögren", "SLE", "MCTD", "IM"]

    # COMMON COMPLETE-CASE ANALYTIC DATASET
    d0 = person_df.copy()
    d0 = d0[d0["ild_status"].isin([0, 1])].copy()
    d0 = d0[d0["male"].isin([0, 1])].copy()
    d0 = d0[d0["ever_smoker"].isin([0, 1])].copy()
    d0 = d0[d0["age_years"].notna()].copy()

    # NO PACK-YEARS
    base_adj_covars = ["age_years", "male", "ever_smoker"]

    exposure_specs = [
        ("Age (per 10 years)", "age_years", True),
        ("Male sex (vs. Female sex)", "male", False),
        ("Ever Smoking (vs. Never Smoking)", "ever_smoker", False),
    ]

    out = []
    for exp_label, exp_col, per10 in exposure_specs:
        out.append({"Exposure": exp_label, "Unadjusted OR (95%CI)": "", "Multivariable adjusted OR (95%CI)": ""})

        for dis in strata:
            dsub = d0[d0["disease"] == dis].copy()
            n_cases = int((dsub["ild_status"] == 1).sum())
            n_ctrl  = int((dsub["ild_status"] == 0).sum())

            # Suppress if either group < 20 (AoU rule)
            if n_cases < min_group_n or n_ctrl < min_group_n:
                unadj, adj = "", ""
            else:
                unadj, adj = _unadj_adj(dsub, exp_col, base_adj_covars, per10=per10)

            if dis == "All SARDs":
                comp = "SARD-ILD (vs. SARD-noILD)"
            else:
                name = DISPLAY[dis]
                comp = f"{name}-ILD (vs. {name}-noILD)"

            out.append({
                "Exposure": f"   {comp}",
                "Unadjusted OR (95%CI)": unadj,
                "Multivariable adjusted OR (95%CI)": adj
            })

        # spacer row between sections
        out.append({"Exposure": "", "Unadjusted OR (95%CI)": "", "Multivariable adjusted OR (95%CI)": ""})

    if out and out[-1]["Exposure"] == "":
        out = out[:-1]

    return pd.DataFrame(out)

table1_shell = build_table1_shell(person_df, min_group_n=20)
table1_shell


NameError: name 'person_df' is not defined

In [26]:
from textwrap import dedent
import pandas_gbq
import pandas as pd

def build_table2_person_df_sql_no_pack(final_sql: str) -> str:
    """
    One row per PERSON:
      - ild_status (1=ILD loose, 0=no ILD codes ever)
      - age_years at earliest SARD index date
      - male (1/0)
      - ever_smoker (1/0; Yes wins)
      - overlapping SARD membership indicators:
          has_RA, has_SLE, has_SSc, has_Sjogren, has_IM, has_MCTD

    IMPORTANT:
      Participants who meet multiple SARD algorithms remain members of all
      applicable SARD subtype indicators.
    """
    return dedent(f"""
    WITH
    cohort AS (
      {final_sql}
    ),

    -- Per-disease index dates
    index_by_person AS (
      SELECT disease, person_id, MIN(DATE(condition_start_datetime)) AS index_date
      FROM cohort
      WHERE disease IN ('RA','SLE','SSc','Sjögren','IM','MCTD')
      GROUP BY disease, person_id
    ),

    -- Earliest SARD index date per person (for age anchoring)
    person_index AS (
      SELECT
        person_id,
        MIN(index_date) AS index_date
      FROM index_by_person
      GROUP BY person_id
    ),

    -- Overlapping SARD membership flags
    sard_flags AS (
      SELECT
        person_id,
        MAX(CASE WHEN disease = 'RA' THEN 1 ELSE 0 END) AS has_RA,
        MAX(CASE WHEN disease = 'SLE' THEN 1 ELSE 0 END) AS has_SLE,
        MAX(CASE WHEN disease = 'SSc' THEN 1 ELSE 0 END) AS has_SSc,
        MAX(CASE WHEN disease = 'Sjögren' THEN 1 ELSE 0 END) AS has_Sjogren,
        MAX(CASE WHEN disease = 'IM' THEN 1 ELSE 0 END) AS has_IM,
        MAX(CASE WHEN disease = 'MCTD' THEN 1 ELSE 0 END) AS has_MCTD
      FROM index_by_person
      GROUP BY person_id
    ),

    -- ILD case definition (loose)
    ild_cases AS (
      {ILD_RELAXED_PEOPLE_SQL}
    ),

    -- no-ILD-ever control definition
    any_ild AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {ILD_UNNEST}
        AND is_standard = 0
    ),
    no_ild_controls AS (
      SELECT DISTINCT c.person_id
      FROM cohort c
      WHERE c.person_id NOT IN (SELECT person_id FROM any_ild)
    ),

    analysis_rows AS (
      SELECT
        pi.person_id,
        pi.index_date,
        sf.has_RA,
        sf.has_SLE,
        sf.has_SSc,
        sf.has_Sjogren,
        sf.has_IM,
        sf.has_MCTD,
        CASE
          WHEN pi.person_id IN (SELECT person_id FROM ild_cases) THEN 1
          WHEN pi.person_id IN (SELECT person_id FROM no_ild_controls) THEN 0
          ELSE NULL
        END AS ild_status
      FROM person_index pi
      JOIN sard_flags sf USING (person_id)
    ),

    -- Demographics
    p AS (
      SELECT person_id, birth_datetime, sex_at_birth_concept_id
      FROM `{CDR}.person`
    ),
    sex_map AS (
      SELECT concept_id, concept_name
      FROM `{CDR}.concept`
    ),

    -- Ever smoker (aggregated: Yes wins; else No; else NULL)
    smoking_map AS (
      SELECT
        person_id,
        CASE
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1 ELSE 0 END) = 1 THEN 1
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: No'  THEN 1 ELSE 0 END) = 1 THEN 0
          ELSE NULL
        END AS ever_smoker
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1585857
      GROUP BY person_id
    )

    SELECT
      r.person_id,
      r.ild_status,

      r.has_RA,
      r.has_SLE,
      r.has_SSc,
      r.has_Sjogren,
      r.has_IM,
      r.has_MCTD,

      -- continuous age in years at earliest SARD index date
      SAFE_DIVIDE(DATE_DIFF(r.index_date, DATE(p.birth_datetime), DAY), 365.25) AS age_years,

      CASE
        WHEN LOWER(cm.concept_name) = 'male'   THEN 1
        WHEN LOWER(cm.concept_name) = 'female' THEN 0
        ELSE NULL
      END AS male,

      sm.ever_smoker
    FROM analysis_rows r
    JOIN p USING (person_id)
    LEFT JOIN sex_map cm ON cm.concept_id = p.sex_at_birth_concept_id
    LEFT JOIN smoking_map sm USING (person_id)
    WHERE r.ild_status IS NOT NULL
    """)

TABLE2_PERSON_SQL = build_table2_person_df_sql_no_pack(final_sql_materialized)
table2_person_df = pandas_gbq.read_gbq(TABLE2_PERSON_SQL, dialect="standard")
print("Table 2 person-level rows (no pack-years):", len(table2_person_df))
table2_person_df.head()

Downloading: 100%|██████████|
Table 2 person-level rows (no pack-years): 9188


,person_id,ild_status,has_RA,has_SLE,has_SSc,has_Sjogren,has_IM,has_MCTD,age_years,male,ever_smoker
0,8415846,0,1,0,0,0,0,0,60.802190,<NA>,0
1,2729401,0,1,0,0,0,0,0,67.561944,<NA>,0
2,8785625,0,1,0,0,0,0,0,69.240246,<NA>,1
3,3508778,0,1,0,0,0,0,0,64.336756,<NA>,0
4,1001685,0,0,0,1,0,0,0,29.481177,<NA>,1


In [32]:
# Run Regression
def _fmt_or_ci(or_val, lo, hi):
    if or_val is None or lo is None or hi is None:
        return ""
    return f"{or_val:.2f} ({lo:.2f}, {hi:.2f})"

def _or_ci(model, term):
    try:
        beta = model.params[term]
        lo, hi = model.conf_int().loc[term].tolist()
        return float(np.exp(beta)), float(np.exp(lo)), float(np.exp(hi))
    except Exception:
        return None, None, None

def _fit_logit_matrix(df, y_col, X_df):
    d = df[[y_col]].join(X_df).dropna().copy()
    if d.empty or d[y_col].nunique() < 2:
        return None
    yv = d[y_col].astype(float)
    X = sm.add_constant(d.drop(columns=[y_col]).astype(float), has_constant="add")
    try:
        return sm.Logit(yv, X).fit(disp=0)
    except Exception:
        return None

def _design_matrix_no_pack(df, *, per10_age=True, include_sard=True):
    d = df.copy()
    d["age10"] = d["age_years"] / 10.0 if per10_age else d["age_years"]

    X = pd.DataFrame({
        "age10": d["age10"],
        "male": d["male"],
        "ever_smoker": d["ever_smoker"],
    })

    if include_sard:
        sard_flags = pd.DataFrame({
            "has_RA": d["has_RA"],
            "has_SLE": d["has_SLE"],
            "has_SSc": d["has_SSc"],
            "has_Sjogren": d["has_Sjogren"],
            "has_IM": d["has_IM"],
            "has_MCTD": d["has_MCTD"],
        })

        # Drop one reference indicator to avoid perfect multicollinearity.
        # RA is the reference.
        sard_flags = sard_flags.drop(columns=["has_RA"])

        X = pd.concat([X, sard_flags], axis=1)

    return X

def _unadj_adj_table2_no_pack(df, exposure_key, *, min_group_n=20):
    """
    exposure_key in {"age","male","ever_smoker"}
    Unadjusted: ild ~ exposure
    Adjusted:   ild ~ age + male + ever_smoker + overlapping SARD indicators
    """
    n_cases = int((df["ild_status"] == 1).sum())
    n_ctrl  = int((df["ild_status"] == 0).sum())
    if n_cases < min_group_n or n_ctrl < min_group_n:
        return "", ""

    if exposure_key == "age":
        X1 = pd.DataFrame({"age10": df["age_years"] / 10.0})
        term1 = "age10"
    else:
        X1 = pd.DataFrame({exposure_key: df[exposure_key]})
        term1 = exposure_key

    m1 = _fit_logit_matrix(df, "ild_status", X1)
    unadj = _fmt_or_ci(*_or_ci(m1, term1)) if m1 is not None else ""

    X2 = _design_matrix_no_pack(df, per10_age=True, include_sard=True)
    term2 = "age10" if exposure_key == "age" else exposure_key

    m2 = _fit_logit_matrix(df, "ild_status", X2)
    adj = _fmt_or_ci(*_or_ci(m2, term2)) if m2 is not None else ""

    return unadj, adj

def build_table2_no_pack(df, min_group_n=20):
    d = df.copy()

    # SAME COMMON COMPLETE-CASE ANALYTIC DATASET AS TABLE 1
    d = d[d["ild_status"].isin([0, 1])].copy()
    d = d[d["male"].isin([0, 1])].copy()
    d = d[d["ever_smoker"].isin([0, 1])].copy()
    d = d[d["age_years"].notna()].copy()

    rows = []

    rows.append({"Risk factor": "Age (per 10 years)", "Unadjusted OR (95%CI)": "", "Multivariable* adjusted OR (95%CI)": ""})
    u, a = _unadj_adj_table2_no_pack(d, "age", min_group_n=min_group_n)
    rows.append({"Risk factor": "", "Unadjusted OR (95%CI)": u, "Multivariable* adjusted OR (95%CI)": a})

    rows.append({"Risk factor": "Male Sex", "Unadjusted OR (95%CI)": "", "Multivariable* adjusted OR (95%CI)": ""})
    u, a = _unadj_adj_table2_no_pack(d, "male", min_group_n=min_group_n)
    rows.append({"Risk factor": "", "Unadjusted OR (95%CI)": u, "Multivariable* adjusted OR (95%CI)": a})

    rows.append({"Risk factor": "Smoking status (ever vs. never)", "Unadjusted OR (95%CI)": "", "Multivariable* adjusted OR (95%CI)": ""})
    u, a = _unadj_adj_table2_no_pack(d, "ever_smoker", min_group_n=min_group_n)
    rows.append({"Risk factor": "", "Unadjusted OR (95%CI)": u, "Multivariable* adjusted OR (95%CI)": a})

    return pd.DataFrame(rows)

table2_no_pack = build_table2_no_pack(table2_person_df, min_group_n=20)
table2_no_pack


NameError: name 'table2_person_df' is not defined

In [40]:
    # Export both to CSV

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
regression_table1 = f"regression_table1_{timestamp}.csv"
regression_table2 = f"regression_table2_{timestamp}.csv"

table1_shell.to_csv(regression_table1, index=False)
table2_no_pack.to_csv(regression_table2, index=False)

print(f"CSV written: {regression_table1}")
print(f"CSV written: {regression_table2}")

NameError: name 'datetime' is not defined

## Logistic Regression for Positive Controls
The following cells perform the same legistic regression on two positive control cohorts, IPF and lung cancer. The negative control cohort consists of All of Us participants with no SARD, ILD, IPF, or lung cancer codes.

In [25]:
# -----------------------------
# Positive controls definitions
# -----------------------------
#  - IPF: 1+ code J84.112 or 516.31 (ICD10CM/ICD9CM)
#  - Lung cancer: 2+ codes on different days (no time limit)
#
# UPDATED NEGATIVE CONTROLS (your new requirement):
#   Negative controls must have:
#     - NO SARD codes ever
#     - NO ILD codes ever
#     - NO IPF codes ever
#     - NO lung cancer codes ever
#
# IMPORTANT (unchanged):
#   - Do NOT exclude IPF from lung cancer analysis or vice versa (i.e., cases are defined only by the outcome rule).

# IPF roots (exact codes)
IPF_ROOTS = ["J84.112", "516.31"]
IPF_CONCEPT_IDS = get_cohort_builder_concept_ids_from_icd_roots(IPF_ROOTS)
IPF_UNNEST = "UNNEST([" + ",".join(str(int(x)) for x in IPF_CONCEPT_IDS) + "])"

# Lung cancer roots (deduped)
LUNG_CANCER_ROOTS = [
    # ICD-10 malignant neoplasm of trachea/bronchus/lung
    "C33", "C34.x",
    # ICD-10 carcinoma in situ of bronchus/lung
    "D02.2", "D02.20", "D02.21", "D02.22",
    # history code you had (ICD-9 V-code)
    "V10.11",
    # ICD-9 malignant neoplasm of trachea/bronchus/lung (use wildcard)
    "162.x",
    # ICD-9 carcinoma in situ of respiratory system
    "231.2",
]
LUNG_CANCER_CONCEPT_IDS = get_cohort_builder_concept_ids_from_icd_roots(LUNG_CANCER_ROOTS)
LUNG_CANCER_UNNEST = "UNNEST([" + ",".join(str(int(x)) for x in LUNG_CANCER_CONCEPT_IDS) + "])"

# SARD concept IDs = union of all expanded ICD roots you already built in Cells 8–10
SARD_CONCEPT_IDS = sorted({int(cid) for lst in _expanded.values() for cid in lst})
SARD_UNNEST = "UNNEST([" + ",".join(str(int(x)) for x in SARD_CONCEPT_IDS) + "])"

print("IPF concept_id count:", len(IPF_CONCEPT_IDS))
print("Lung cancer concept_id count:", len(LUNG_CANCER_CONCEPT_IDS))
print("SARD concept_id count (union across diseases):", len(SARD_CONCEPT_IDS))

# NOTE:
#   ILD_UNNEST is assumed to already exist in your notebook (used earlier for NO_ILD_SQL).
#   If it doesn't, define it the same way you defined it for the ILD tables.

NameError: name 'get_cohort_builder_concept_ids_from_icd_roots' is not defined

In [24]:
def build_positive_control_person_df_sql_no_pack(
    *,
    outcome_name: str,
    outcome_unnest: str,
    case_rule: str,  # "ipf_1plus" or "lung_cancer_2days"
    final_sard_sql: str,
    # NEW: negative control exclusions
    sard_unnest: str,
    ild_unnest: str,
    ipf_unnest: str,
    lung_cancer_unnest: str,
) -> str:
    """
    Builds ONE person-level dataset for a positive-control outcome:

      outcome_status:
        - 1 = case (meets outcome definition)
        - 0 = negative control (meets STRICT negative control definition)

      covariates:
        - age_years at AoU enrollment (primary consent date)
        - male (1/0)
        - ever_smoker (1/0; Yes wins)

    STRICT negative controls (your requirement):
      - NO SARD codes ever
      - NO ILD codes ever
      - NO IPF codes ever
      - NO lung cancer codes ever
      - AND NO outcome ever (redundant for IPF/lung cancer outcomes, but kept explicit)
      - AND consented to EHR data collection
    """
    assert case_rule in ("ipf_1plus", "lung_cancer_2days")

    # Case people SQL:
    if case_rule == "ipf_1plus":
        cases_cte = f"""
        outcome_events AS (
          SELECT person_id, DATE(entry_date) AS dt
          FROM `{CDR}.cb_search_all_events`
          WHERE concept_id IN {outcome_unnest}
            AND is_standard = 0
        ),
        outcome_first AS (
          SELECT person_id, MIN(dt) AS first_dt
          FROM outcome_events
          GROUP BY person_id
        ),
        cases AS (
          SELECT person_id, first_dt AS outcome_index_date
          FROM outcome_first
        )
        """
    else:
        cases_cte = f"""
        outcome_events AS (
          SELECT person_id, DATE(entry_date) AS dt
          FROM `{CDR}.cb_search_all_events`
          WHERE concept_id IN {outcome_unnest}
            AND is_standard = 0
        ),
        outcome_first AS (
          SELECT person_id, MIN(dt) AS first_dt
          FROM outcome_events
          GROUP BY person_id
        ),
        cases AS (
          SELECT f.person_id, f.first_dt AS outcome_index_date
          FROM outcome_first f
          WHERE EXISTS (
            SELECT 1
            FROM outcome_events e
            WHERE e.person_id = f.person_id
              AND e.dt >= DATE_ADD(f.first_dt, INTERVAL 1 DAY)
          )
        )
        """

    return dedent(f"""
    WITH
    -- SARD cohort members (kept for reference; no longer the control rule)
    sard_people AS (
      SELECT DISTINCT person_id
      FROM ({final_sard_sql})
    ),

    {cases_cte},

    -- EHR-consented participants (for defining controls)
    ehr_consented AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.observation`
      WHERE observation_source_concept_id = 1586099
        AND value_source_concept_id = 1586100
    ),

    -- Any outcome ever (for defining controls)
    any_outcome AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {outcome_unnest}
        AND is_standard = 0
    ),

    -- NEW: Any SARD / ILD / IPF / Lung cancer ever (for strict negative controls)
    any_sard AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {sard_unnest}
        AND is_standard = 0
    ),
    any_ild AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {ild_unnest}
        AND is_standard = 0
    ),
    any_ipf AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {ipf_unnest}
        AND is_standard = 0
    ),
    any_lung_cancer AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {lung_cancer_unnest}
        AND is_standard = 0
    ),

    -- STRICT negative controls
    controls AS (
      SELECT p.person_id
      FROM `{CDR}.person` p
      WHERE p.person_id IN (SELECT person_id FROM ehr_consented)
        AND p.person_id NOT IN (SELECT person_id FROM any_outcome)
        AND p.person_id NOT IN (SELECT person_id FROM any_sard)
        AND p.person_id NOT IN (SELECT person_id FROM any_ild)
        AND p.person_id NOT IN (SELECT person_id FROM any_ipf)
        AND p.person_id NOT IN (SELECT person_id FROM any_lung_cancer)
    ),

    analysis_rows AS (
      SELECT person_id, 1 AS outcome_status
      FROM cases
      UNION ALL
      SELECT person_id, 0 AS outcome_status
      FROM controls
    ),

    -- AoU enrollment proxy: primary consent date (per AoU guidance)
    primary_consent AS (
      SELECT
        o.person_id,
        MIN(o.observation_date) AS primary_consent_date
      FROM `{CDR}.concept` c
      JOIN `{CDR}.concept_ancestor` ca
        ON c.concept_id = ca.ancestor_concept_id
      JOIN `{CDR}.observation` o
        ON ca.descendant_concept_id = o.observation_concept_id
      WHERE c.concept_name = 'Consent PII'
        AND c.concept_class_id = 'Module'
      GROUP BY o.person_id
    ),

    -- Demographics
    p AS (
      SELECT person_id, birth_datetime, sex_at_birth_concept_id
      FROM `{CDR}.person`
    ),
    sex_map AS (
      SELECT concept_id, concept_name
      FROM `{CDR}.concept`
    ),

    -- Ever smoker (Yes wins; else No; else NULL)
    smoking_map AS (
      SELECT
        person_id,
        CASE
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1 ELSE 0 END) = 1 THEN 1
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: No'  THEN 1 ELSE 0 END) = 1 THEN 0
          ELSE NULL
        END AS ever_smoker
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1585857
      GROUP BY person_id
    )

    SELECT
      '{outcome_name}' AS outcome,
      a.person_id,
      a.outcome_status,

      pc.primary_consent_date AS enrollment_date,

      SAFE_DIVIDE(DATE_DIFF(pc.primary_consent_date, DATE(p.birth_datetime), DAY), 365.25) AS age_years,

      CASE
        WHEN LOWER(cm.concept_name) = 'male'   THEN 1
        WHEN LOWER(cm.concept_name) = 'female' THEN 0
        ELSE NULL
      END AS male,

      sm.ever_smoker

    FROM analysis_rows a
    JOIN p USING (person_id)
    JOIN primary_consent pc USING (person_id)
    LEFT JOIN sex_map cm ON cm.concept_id = p.sex_at_birth_concept_id
    LEFT JOIN smoking_map sm USING (person_id)
    WHERE pc.primary_consent_date IS NOT NULL
    """)

In [23]:
def run_positive_control_no_pack(outcome_name, outcome_unnest, case_rule, min_group_n=20):
    SQL = build_positive_control_person_df_sql_no_pack(
        outcome_name=outcome_name,
        outcome_unnest=outcome_unnest,
        case_rule=case_rule,
        final_sard_sql=final_sql_materialized,

        # NEW: strict negative control exclusions
        sard_unnest=SARD_UNNEST,
        ild_unnest=ILD_UNNEST,
        ipf_unnest=IPF_UNNEST,
        lung_cancer_unnest=LUNG_CANCER_UNNEST,
    )
    df = pandas_gbq.read_gbq(SQL, dialect="standard")
    print(outcome_name, "rows:", len(df), "| cases:", int((df["outcome_status"]==1).sum()), "| ctrls:", int((df["outcome_status"]==0).sum()))

    # clean contrasts like your Table 2 code
    d = df.copy()
    d = d[d["outcome_status"].isin([0, 1])].copy()
    d = d[d["male"].isin([0, 1])].copy()
    d = d[d["ever_smoker"].isin([0, 1])].copy()
    d = d[d["age_years"].notna()].copy()

    n_cases = int((d["outcome_status"] == 1).sum())
    n_ctrl  = int((d["outcome_status"] == 0).sum())
    if n_cases < min_group_n or n_ctrl < min_group_n:
        print("Suppressed: <20 in a group.")
        return df, pd.DataFrame()

    def _fit_logit_matrix_y(df0, y_col, X_df):
        dd = df0[[y_col]].join(X_df).dropna().copy()
        if dd.empty or dd[y_col].nunique() < 2:
            return None
        yv = dd[y_col].astype(float)
        X  = sm.add_constant(dd.drop(columns=[y_col]).astype(float), has_constant="add")
        try:
            return sm.Logit(yv, X).fit(disp=0)
        except Exception:
            return None

    def _or_ci(model, term):
        try:
            beta = model.params[term]
            lo, hi = model.conf_int().loc[term].tolist()
            return float(np.exp(beta)), float(np.exp(lo)), float(np.exp(hi))
        except Exception:
            return None, None, None

    def _fmt_or_ci(or_val, lo, hi):
        if or_val is None or lo is None or hi is None:
            return ""
        return f"{or_val:.2f} ({lo:.2f}, {hi:.2f})"

    def _unadj_adj(exposure_key):
        # unadjusted
        if exposure_key == "age":
            X1 = pd.DataFrame({"age10": d["age_years"] / 10.0})
            term1 = "age10"
        else:
            X1 = pd.DataFrame({exposure_key: d[exposure_key]})
            term1 = exposure_key
        m1 = _fit_logit_matrix_y(d, "outcome_status", X1)
        unadj = _fmt_or_ci(*_or_ci(m1, term1)) if m1 is not None else ""

        # adjusted: age + male + ever_smoker
        X2 = pd.DataFrame({
            "age10": d["age_years"] / 10.0,
            "male": d["male"],
            "ever_smoker": d["ever_smoker"],
        })
        term2 = "age10" if exposure_key == "age" else exposure_key
        m2 = _fit_logit_matrix_y(d, "outcome_status", X2)
        adj = _fmt_or_ci(*_or_ci(m2, term2)) if m2 is not None else ""
        return unadj, adj

    rows = []
    rows.append({"Risk factor": "Age (per 10 years)", "Unadjusted OR (95%CI)": "", "Multivariable adjusted OR (95%CI)": ""})
    u,a = _unadj_adj("age"); rows.append({"Risk factor":"", "Unadjusted OR (95%CI)":u, "Multivariable adjusted OR (95%CI)":a})

    rows.append({"Risk factor": "Male Sex", "Unadjusted OR (95%CI)": "", "Multivariable adjusted OR (95%CI)": ""})
    u,a = _unadj_adj("male"); rows.append({"Risk factor":"", "Unadjusted OR (95%CI)":u, "Multivariable adjusted OR (95%CI)":a})

    rows.append({"Risk factor": "Smoking status (ever vs. never)", "Unadjusted OR (95%CI)": "", "Multivariable adjusted OR (95%CI)": ""})
    u,a = _unadj_adj("ever_smoker"); rows.append({"Risk factor":"", "Unadjusted OR (95%CI)":u, "Multivariable adjusted OR (95%CI)":a})

    out = pd.DataFrame(rows)
    return df, out

# Run
ipf_df_no_pack, ipf_table_no_pack = run_positive_control_no_pack("IPF", IPF_UNNEST, "ipf_1plus", min_group_n=20)
lung_df_no_pack, lung_table_no_pack = run_positive_control_no_pack("Lung cancer", LUNG_CANCER_UNNEST, "lung_cancer_2days", min_group_n=20)

ipf_table_no_pack, lung_table_no_pack


NameError: name 'IPF_UNNEST' is not defined

In [21]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

ipf_table_no_pack.to_csv(f"positive_control_ipf_table2_{timestamp}.csv", index=False)
lung_table_no_pack.to_csv(f"positive_control_lung_cancer_table2_{timestamp}.csv", index=False)


print("Wrote 2 positive-control regression CSVs with timestamp", timestamp)

NameError: name 'datetime' is not defined

In [42]:
# ============================================================
# Summary table builder for POSITIVE CONTROLS + STRICT NEGATIVE CONTROLS
#   Columns: IPF, Lung cancer, Negative controls
#   Rows:    same "core" rows as build_sard_summary_table
#            EXCEPT: no meds section, no ILD section
#   Index date anchor: AoU enrollment (primary consent date) for all groups
#
# UPDATED to match regression code (PC2/PC4) negative controls:
#   Strict negative controls must have:
#     - NO SARD codes ever           (event-based; concept_id IN SARD_UNNEST; is_standard=0)
#     - NO ILD codes ever            (concept_id IN ILD_UNNEST;  is_standard=0)
#     - NO IPF codes ever            (concept_id IN IPF_UNNEST;  is_standard=0)
#     - NO lung cancer codes ever    (concept_id IN LUNG_CANCER_UNNEST; is_standard=0)
#     - EHR consented in AoU         (observation_source_concept_id = 1586099
#                                     and value_source_concept_id = 1586100)
#
# REQUIREMENTS (must already exist in your notebook):
#   - CDR
#   - final_sql_materialized (kept for compatibility; not used for negative controls anymore)
#   - ILD_UNNEST
#   - IPF_UNNEST
#   - LUNG_CANCER_UNNEST
#   - SARD_UNNEST   <-- NEW (must match the same SARD_UNNEST used in PC2/PC4)
# ============================================================

import re
import numpy as np
import pandas as pd
import pandas_gbq
from datetime import datetime
from textwrap import dedent


def build_controls_person_level_sql(
    *,
    final_sard_sql: str,  # kept for compatibility; not used for strict negatives anymore
    ipf_unnest: str,
    lung_cancer_unnest: str,
    ild_unnest: str,
    sard_unnest: str,  # NEW
) -> str:
    """
    One row per person with:
      - group_label in ('IPF','Lung cancer','Negative controls')
      - enrollment_date (primary consent) as index_date
      - age_years at enrollment
      - sex (Male/Female/Unknown)
      - race (self-reported; missing bucket)
      - ever_smoker (1/0/NULL; Yes wins)
      - packyears (float; ONLY computed for ever_smoker=1; NULL if missing components)

    STRICT Negative controls (MATCHES PC2/PC4):
      - NO SARD codes ever (concept_id IN sard_unnest, is_standard=0)
      - NO ILD codes ever  (concept_id IN ild_unnest,  is_standard=0)
      - NO IPF codes ever  (concept_id IN ipf_unnest,  is_standard=0)
      - NO lung cancer codes ever (concept_id IN lung_cancer_unnest, is_standard=0)
      - EHR consented in AoU

    Cases:
      - IPF: 1+ code ever
      - Lung cancer: 2+ codes on different days (no time limit)
    """
    return dedent(f"""
    WITH
    -- ---- Outcome events ----
    ipf_events AS (
      SELECT person_id, DATE(entry_date) AS dt
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {ipf_unnest}
        AND is_standard = 0
    ),
    ipf_cases AS (
      SELECT DISTINCT person_id
      FROM ipf_events
    ),

    lung_events AS (
      SELECT person_id, DATE(entry_date) AS dt
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {lung_cancer_unnest}
        AND is_standard = 0
    ),
    lung_first AS (
      SELECT person_id, MIN(dt) AS first_dt
      FROM lung_events
      GROUP BY person_id
    ),
    lung_cases AS (
      SELECT f.person_id
      FROM lung_first f
      WHERE EXISTS (
        SELECT 1
        FROM lung_events e
        WHERE e.person_id = f.person_id
          AND e.dt >= DATE_ADD(f.first_dt, INTERVAL 1 DAY)
      )
    ),

    -- EHR-consented participants
    ehr_consented AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.observation`
      WHERE observation_source_concept_id = 1586099
        AND value_source_concept_id = 1586100
    ),

    -- Any forbidden buckets ever (MATCH regression code)
    any_sard AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {sard_unnest}
        AND is_standard = 0
    ),
    any_ild AS (
      SELECT DISTINCT person_id
      FROM `{CDR}.cb_search_all_events`
      WHERE concept_id IN {ild_unnest}
        AND is_standard = 0
    ),
    any_ipf AS (
      SELECT DISTINCT person_id
      FROM ipf_events
    ),
    any_lung AS (
      SELECT DISTINCT person_id
      FROM lung_events
    ),

    -- AoU enrollment proxy: primary consent date
    primary_consent AS (
      SELECT
        o.person_id,
        MIN(o.observation_date) AS primary_consent_date
      FROM `{CDR}.concept` c
      JOIN `{CDR}.concept_ancestor` ca
        ON c.concept_id = ca.ancestor_concept_id
      JOIN `{CDR}.observation` o
        ON ca.descendant_concept_id = o.observation_concept_id
      WHERE c.concept_name = 'Consent PII'
        AND c.concept_class_id = 'Module'
      GROUP BY o.person_id
    ),

    -- Strict negative controls universe: persons with consent date, EHR consent,
    -- and excluding all forbidden disease buckets
    strict_negative_controls AS (
      SELECT p.person_id
      FROM `{CDR}.person` p
      JOIN primary_consent pc ON pc.person_id = p.person_id
      WHERE p.person_id IN (SELECT person_id FROM ehr_consented)
        AND p.person_id NOT IN (SELECT person_id FROM any_sard)
        AND p.person_id NOT IN (SELECT person_id FROM any_ild)
        AND p.person_id NOT IN (SELECT person_id FROM any_ipf)
        AND p.person_id NOT IN (SELECT person_id FROM any_lung)
    ),

    cohort_rows AS (
      SELECT person_id, 'IPF' AS group_label FROM ipf_cases
      UNION ALL
      SELECT person_id, 'Lung cancer' AS group_label FROM lung_cases
      UNION ALL
      SELECT person_id, 'Negative controls' AS group_label FROM strict_negative_controls
    ),

    -- Demographics
    p AS (
      SELECT person_id, birth_datetime, COALESCE(sex_at_birth_concept_id, 0) AS sex_id
      FROM `{CDR}.person`
    ),
    concept_map AS (
      SELECT concept_id, concept_name
      FROM `{CDR}.concept`
    ),
    sex_labeled AS (
      SELECT
        p.person_id,
        CASE
          WHEN LOWER(cm.concept_name) = 'male' THEN 'Male'
          WHEN LOWER(cm.concept_name) = 'female' THEN 'Female'
          ELSE 'Unknown'
        END AS sex
      FROM p
      LEFT JOIN concept_map cm ON cm.concept_id = p.sex_id
    ),

    -- Race (self-reported)
    race_raw AS (
      SELECT person_id, answer AS race
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1586140
    ),
    race_labeled AS (
      SELECT
        person_id,
        COALESCE(race, 'Prefer not to answer / missing') AS race
      FROM race_raw
    ),

    -- Ever smoker (Yes wins; else No; else NULL)
    smoking_map AS (
      SELECT
        person_id,
        CASE
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1 ELSE 0 END) = 1 THEN 1
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: No'  THEN 1 ELSE 0 END) = 1 THEN 0
          ELSE NULL
        END AS ever_smoker
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1585857
      GROUP BY person_id
    ),

    -- Pack-years components (Lifestyle survey)
    lifestyle AS (
      SELECT person_id, question, answer
      FROM `{CDR}.ds_survey`
      WHERE survey LIKE '%Lifestyle%'
    ),
    avg_daily AS (
      SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS cigs_per_day
      FROM lifestyle
      WHERE question = 'Smoking: Average Daily Cigarette Number'
    ),
    years_smoked AS (
      SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS years
      FROM lifestyle
      WHERE question = 'Smoking: Number Of Years'
    ),
    packyears_map AS (
      SELECT
        sm.person_id,
        (ys.years * (ad.cigs_per_day / 20.0)) AS packyears
      FROM smoking_map sm
      LEFT JOIN avg_daily ad    ON ad.person_id = sm.person_id
      LEFT JOIN years_smoked ys ON ys.person_id = sm.person_id
      WHERE sm.ever_smoker = 1
    )

    SELECT
      cr.group_label,
      cr.person_id,
      pc.primary_consent_date AS index_date,

      SAFE_DIVIDE(DATE_DIFF(pc.primary_consent_date, DATE(p.birth_datetime), DAY), 365.25) AS age_years,

      sx.sex,
      COALESCE(rc.race, 'Prefer not to answer / missing') AS race,

      sm.ever_smoker,
      py.packyears
    FROM cohort_rows cr
    JOIN primary_consent pc USING (person_id)
    JOIN p USING (person_id)
    LEFT JOIN sex_labeled sx USING (person_id)
    LEFT JOIN race_labeled rc USING (person_id)
    LEFT JOIN smoking_map sm USING (person_id)
    LEFT JOIN packyears_map py USING (person_id)
    WHERE pc.primary_consent_date IS NOT NULL
    """)


def _fmt_mean_sd(x: pd.Series) -> str:
    x = pd.to_numeric(x, errors="coerce").dropna()
    if x.empty:
        return ""
    return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"


def _fmt_n_pct(n: int, denom: int) -> str:
    if denom <= 0:
        return ""
    return f"{int(n)} ({100.0 * n / denom:.1f}%)"


def _fmt_suppressed_if_needed(n: int, denom: int) -> str:
    """
    AoU suppression: if numerator < 20, return '< 20 (< X.X%)' where X.X is 20/denom*100.
    """
    if denom <= 0:
        return "< 20"
    if n < 20:
        pct20 = 100.0 * 20.0 / float(denom)
        return f"< 20 (< {pct20:.1f}%)"
    return _fmt_n_pct(n, denom)


def build_controls_summary_table(
    *,
    final_sard_sql: str,  # kept for backward compatibility
    ipf_unnest: str,
    lung_cancer_unnest: str,
    ild_unnest: str,
    sard_unnest: str,  # NEW
    export_prefix: str = "controls_summary",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - person_level_df
      - summary_table_df (Characteristic + three group columns with n in headers)

    Excludes meds and ILD rows by construction.
    """
    sql = build_controls_person_level_sql(
        final_sard_sql=final_sard_sql,
        ipf_unnest=ipf_unnest,
        lung_cancer_unnest=lung_cancer_unnest,
        ild_unnest=ild_unnest,
        sard_unnest=sard_unnest,
    )
    person = pandas_gbq.read_gbq(sql, dialect="standard")

    # Ensure expected labels/order
    group_order = ["IPF", "Lung cancer", "Negative controls"]
    person["group_label"] = pd.Categorical(person["group_label"], categories=group_order, ordered=True)

    # Denominators
    denom = (
        person.groupby("group_label")["person_id"]
        .nunique()
        .reindex(group_order)
        .fillna(0)
        .astype(int)
        .to_dict()
    )

    # Column headers with n
    col_names = {g: f"{g} (n={denom.get(g, 0)})" for g in group_order}

    # Definition row text
    definition_row = {
        "Characteristic": "Definition",
        col_names["IPF"]: "IPF cases: ≥1 IPF code ever (J84.112 or 516.31 roots expanded)",
        col_names["Lung cancer"]: "Lung cancer cases: ≥2 lung cancer codes on different days (roots expanded)",
        col_names["Negative controls"]: "Strict negatives: EHR-consented; no SARD codes ever; no ILD; no IPF; no lung cancer codes ever",
    }

    rows = [definition_row]

    # AGE
    age_row = {"Characteristic": "Age (years, mean, SD)"}
    for g in group_order:
        age_row[col_names[g]] = _fmt_mean_sd(person.loc[person["group_label"] == g, "age_years"])
    rows.append(age_row)

    # SEX (Female/Male) with suppression
    for sex_label, out_label in [("Female", "Female sex (n, %)"), ("Male", "Male sex (n, %)")]:
        r = {"Characteristic": out_label}
        for g in group_order:
            dsub = person[person["group_label"] == g]
            n = int((dsub["sex"] == sex_label).sum())
            r[col_names[g]] = _fmt_suppressed_if_needed(n, denom[g])
        rows.append(r)

    # RACE header
    rows.append({
        "Characteristic": "Self-reported Race",
        col_names["IPF"]: "",
        col_names["Lung cancer"]: "",
        col_names["Negative controls"]: "",
    })

    # Race levels (union across all groups)
    race_levels = (
        person["race"]
        .fillna("Prefer not to answer / missing")
        .astype(str)
        .unique()
        .tolist()
    )
    race_levels = sorted(race_levels, key=lambda s: s.lower())

    for lvl in race_levels:
        r = {"Characteristic": lvl}
        for g in group_order:
            dsub = person[person["group_label"] == g]
            n = int((dsub["race"].fillna("Prefer not to answer / missing").astype(str) == str(lvl)).sum())
            r[col_names[g]] = _fmt_suppressed_if_needed(n, denom[g])
        rows.append(r)

    # Ever smoker (suppressed)
    r_smoke = {"Characteristic": "Ever smoker (n, %)"}
    for g in group_order:
        dsub = person[person["group_label"] == g]
        n = int((dsub["ever_smoker"] == 1).sum())
        r_smoke[col_names[g]] = _fmt_suppressed_if_needed(n, denom[g])
    rows.append(r_smoke)

    # Pack-years among ever-smokers with non-null packyears (NO suppression for mean/sd)
    r_pack = {"Characteristic": "Pack-years (mean, SD)"}
    for g in group_order:
        dsub = person[person["group_label"] == g].copy()
        dsub = dsub[(dsub["ever_smoker"] == 1) & (~pd.isna(dsub["packyears"]))].copy()
        r_pack[col_names[g]] = _fmt_mean_sd(dsub["packyears"])
    rows.append(r_pack)

    summary = pd.DataFrame(rows)

    # Export
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"{export_prefix}_{ts}.csv"
    summary.to_csv(csv_path, index=False)
    print("Wrote:", csv_path)

    return person, summary


# -----------------
# RUN
# -----------------
controls_person_df, controls_summary_table = build_controls_summary_table(
    final_sard_sql=final_sql_materialized,  # kept for compatibility
    ipf_unnest=IPF_UNNEST,
    lung_cancer_unnest=LUNG_CANCER_UNNEST,
    ild_unnest=ILD_UNNEST,
    sard_unnest=SARD_UNNEST,  # IMPORTANT: must match regression code
    export_prefix="positive_negative_controls_summary",
)

controls_summary_table

ModuleNotFoundError: No module named 'pandas_gbq'

## Miscellaneous
The following cells answer a couple of additional questions about our data, including how many participants are in the All of Us dataset and what percentage of our cohorts we would lose if we included pack-years as an additional variable in our regression models.

In [17]:
# ============================================================
# Total number of participants in All of Us
# ============================================================

AOU_N_SQL = f"""
SELECT COUNT(DISTINCT person_id) AS n_all_of_us_participants
FROM `{CDR}.person`
"""

aou_n_df = pandas_gbq.read_gbq(AOU_N_SQL, dialect="standard")
aou_n_df

NameError: name 'CDR' is not defined

In [18]:
# ============================================================
# Helpers 
# ============================================================

from textwrap import dedent
import pandas as pd
import pandas_gbq

def build_cohort_missingness_sql(cohort_sql: str, cohort_label: str) -> str:
    """
    For a cohort defined by cohort_sql (must include person_id),
    returns one-row summary with:
      - n_cohort
      - n_missing_smoking
      - n_smokers
      - n_smokers_missing_packyears
    """
    return dedent(f"""
    WITH
    cohort AS (
      {cohort_sql}
    ),
    people AS (
      SELECT DISTINCT person_id
      FROM cohort
    ),

    smoking_raw AS (
      SELECT person_id, answer
      FROM `{CDR}.ds_survey`
      WHERE question_concept_id = 1585857
    ),
    smoking_map AS (
      SELECT
        person_id,
        CASE
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: Yes' THEN 1 ELSE 0 END) = 1 THEN 1
          WHEN MAX(CASE WHEN answer='100 Cigs Lifetime: No'  THEN 1 ELSE 0 END) = 1 THEN 0
          ELSE NULL
        END AS ever_smoker
      FROM smoking_raw
      GROUP BY person_id
    ),

    lifestyle AS (
      SELECT person_id, question, answer
      FROM `{CDR}.ds_survey`
      WHERE survey LIKE '%Lifestyle%'
    ),
    avg_daily AS (
      SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS cigs_per_day
      FROM lifestyle
      WHERE question = 'Smoking: Average Daily Cigarette Number'
    ),
    years_smoked AS (
      SELECT person_id, SAFE_CAST(answer AS FLOAT64) AS years
      FROM lifestyle
      WHERE question = 'Smoking: Number Of Years'
    ),
    packyears_map AS (
      SELECT
        sm.person_id,
        (ys.years * (ad.cigs_per_day / 20.0)) AS packyears
      FROM smoking_map sm
      LEFT JOIN avg_daily ad    ON ad.person_id = sm.person_id
      LEFT JOIN years_smoked ys ON ys.person_id = sm.person_id
      WHERE sm.ever_smoker = 1
    )

    SELECT
      '{cohort_label}' AS cohort,
      COUNT(DISTINCT p.person_id) AS n_cohort,
      COUNT(DISTINCT IF(sm.ever_smoker IS NULL, p.person_id, NULL)) AS n_missing_smoking,
      COUNT(DISTINCT IF(sm.ever_smoker = 1, p.person_id, NULL)) AS n_smokers,
      COUNT(DISTINCT IF(sm.ever_smoker = 1 AND py.packyears IS NULL, p.person_id, NULL)) AS n_smokers_missing_packyears
    FROM people p
    LEFT JOIN smoking_map sm USING (person_id)
    LEFT JOIN packyears_map py USING (person_id)
    """)


def build_sard_multiplicity_sql(cohort_sql: str, cohort_label: str) -> str:
    """
    Counts how many people in a SARD-based cohort meet exactly 1, 2, or 3+ SARD algorithms.
    """
    return dedent(f"""
    WITH
    cohort AS (
      {cohort_sql}
    ),
    disease_counts AS (
      SELECT
        person_id,
        COUNT(DISTINCT disease) AS n_sard_diseases
      FROM cohort
      GROUP BY person_id
    )
    SELECT
      '{cohort_label}' AS cohort,
      COUNT(DISTINCT person_id) AS n_people,
      COUNT(DISTINCT IF(n_sard_diseases = 1, person_id, NULL)) AS exactly_1_disease,
      COUNT(DISTINCT IF(n_sard_diseases = 2, person_id, NULL)) AS exactly_2_diseases,
      COUNT(DISTINCT IF(n_sard_diseases >= 3, person_id, NULL)) AS exactly_3plus_diseases
    FROM disease_counts
    """)

ModuleNotFoundError: No module named 'pandas_gbq'

In [15]:
# ============================================================
# Missing smoking data and missing pack-years among smokers
# ============================================================

controls_person_level_sql = build_controls_person_level_sql(
    final_sard_sql=final_sql_materialized,
    ipf_unnest=IPF_UNNEST,
    lung_cancer_unnest=LUNG_CANCER_UNNEST,
    ild_unnest=ILD_UNNEST,
    sard_unnest=SARD_UNNEST,
)

# Run SARD-family cohort missingness queries separately, then stack
sard_missingness_df = pd.concat(
    [
        pandas_gbq.read_gbq(
            build_cohort_missingness_sql(final_sql_materialized, "All SARD"),
            dialect="standard",
        ),
        pandas_gbq.read_gbq(
            build_cohort_missingness_sql(ild_relaxed_cohort_sql, "SARD-ILD (loose)"),
            dialect="standard",
        ),
        pandas_gbq.read_gbq(
            build_cohort_missingness_sql(NO_ILD_SQL, "SARD-noILD"),
            dialect="standard",
        ),
    ],
    ignore_index=True,
)

controls_person_df_for_missingness = pandas_gbq.read_gbq(
    controls_person_level_sql,
    dialect="standard",
)

controls_missingness_df = (
    controls_person_df_for_missingness
    .groupby("group_label", dropna=False)
    .agg(
        n_cohort=("person_id", "nunique"),
        n_missing_smoking=("ever_smoker", lambda x: int(pd.isna(x).sum())),
        n_smokers=("ever_smoker", lambda x: int((x == 1).fillna(False).sum())),
        n_smokers_missing_packyears=("person_id", lambda ids: 0),  # placeholder; overwritten below
    )
    .reset_index()
    .rename(columns={"group_label": "cohort"})
)

_pack_missing = (
    controls_person_df_for_missingness
    .assign(
        smoker_missing_pack=lambda d: (
            ((d["ever_smoker"] == 1) & (d["packyears"].isna()))
            .fillna(False)
            .astype(int)
        )
    )
    .groupby("group_label", dropna=False)["smoker_missing_pack"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "group_label": "cohort",
            "smoker_missing_pack": "n_smokers_missing_packyears",
        }
    )
)

controls_missingness_df = controls_missingness_df.drop(
    columns=["n_smokers_missing_packyears"]
).merge(
    _pack_missing,
    on="cohort",
    how="left",
)

missingness_summary = pd.concat(
    [sard_missingness_df, controls_missingness_df],
    ignore_index=True,
)

missingness_summary

NameError: name 'build_controls_person_level_sql' is not defined